In [2]:
# SPUR DuckDB extension setup cell v1
# requirements: {"active":[{"packageName":"duckrouting","source":"community"},{"packageName":"shellfs","source":"community"}],"pending":[]}
# Managed by Jute. Change packages through the extension sidebar.
# SPUR DuckDB bootstrap
import duckdb
import sys
_SPUR_DUCKDB_EXTENSION_PATH = "/Users/kevintruong/.spur/extensions/spur_rest.duckdb_extension"
# Shared by notebook setup and classic SQL cells. Preserve the generic path for
# older bundles; new bundles carry exact native ABI artifacts per engine.
from pathlib import Path as _SpurExtensionPath

_spur_extension = _SpurExtensionPath(_SPUR_DUCKDB_EXTENSION_PATH)
_spur_versioned_extension = (
    _spur_extension.parent
    / ("duckdb-v" + duckdb.__version__)
    / _spur_extension.name
)
if _spur_versioned_extension.is_file():
    _SPUR_DUCKDB_EXTENSION_PATH = str(_spur_versioned_extension)

if (
    sys.platform in ("linux", "darwin")
    and duckdb.__version__ in ("1.4.4", "1.5.5")
    and "_SPUR_DUCKDB_HOST_SYMBOLS" not in globals()
):
    import ctypes as _spur_ctypes
    import _duckdb as _spur_host_duckdb

    # Promote the already-loaded Python engine's symbols. This opens the same
    # module handle; it neither imports another engine nor replaces connections.
    _SPUR_DUCKDB_HOST_SYMBOLS = _spur_ctypes.CDLL(
        _spur_host_duckdb.__file__, mode=_spur_ctypes.RTLD_GLOBAL
    )

_SPUR_DUCKDB_EXTENSION_SQL = _SPUR_DUCKDB_EXTENSION_PATH.replace("'", "''")

if "_SPUR_DUCKDB_CONNECTION" not in globals():
    _SPUR_DUCKDB_CONNECTION = duckdb.connect(
        database=":memory:",
        config={"allow_unsigned_extensions": "true",
                 "autoload_known_extensions": "false",
                 "autoinstall_known_extensions": "false"},
    )

duckdb.set_default_connection(_SPUR_DUCKDB_CONNECTION)
# Embedded helper source contains no token, credential path or capability lease.
# The approved local launcher passes a private inherited socket descriptor, not
# executable source or a notebook/widget message. Same-kernel code is trusted.
def _spur_managed_connection(db):
    """Only the host's selected connection is subject to managed policy."""
    import types
    connection = db.default_connection() if isinstance(db, types.ModuleType) else db
    if connection is not globals().get("_SPUR_DUCKDB_CONNECTION"):
        return None
    try:
        connection.sql("SET autoload_known_extensions = false")
        connection.sql("SET autoinstall_known_extensions = false")
    except Exception:
        return None  # existing locked/custom connections have an explicit boundary
    return connection


# Required SpurLab baseline: JSON/parquet are statically linked in the managed
# wheel; httpfs supplies remote files and REST gateway dependencies; spur_rest is
# the separately packaged, host-verified gateway (never a repository download).
# Remote datasource attach code explicitly installs/loads its provider adapter.
# Spatial and Jev are optional. No browsing/query path calls operate("install").
# Fixed metadata query only. Identifiers and descriptions remain inert JSON; no
# catalog value is interpolated as SQL. The limits bound the shell reply too.
def _spur_function_catalog(db):
    import json
    limit = 10000
    rows = db.sql("SELECT database_name, schema_name, function_name, function_type, "
                  "parameters, parameter_types, varargs, return_type, description, examples, function_oid "
                  "FROM duckdb_functions() ORDER BY database_name, schema_name, function_name, "
                  "function_type, parameter_types, varargs LIMIT 10001").fetchall()
    functions, size = [], 0
    complete = len(rows) <= limit
    for row in rows[:limit]:
        value = dict(zip(("database_name", "schema_name", "name", "function_type", "parameters",
                          "parameter_types", "varargs", "return_type", "description", "examples", "function_oid"), row))
        value["ownership"] = "unknown"
        encoded = json.dumps(value, ensure_ascii=True)
        # Oversized metadata is omitted, with incomplete coverage made explicit.
        if len(encoded) > 65536 or size + len(encoded) > 4 * 1024 * 1024:
            complete = False
            continue
        functions.append(value)
        size += len(encoded)
    return {"status": "complete" if complete else "incomplete", "functions": functions,
            "truncated": not complete}


def _spur_function_identity(value):
    return (value["database_name"], value["schema_name"], value["name"],
            value["function_type"], tuple(value["parameter_types"]), value["varargs"])


def _spur_snapshot_function_registration(db):
    try:
        snapshot = _spur_function_catalog(db)
        if snapshot["status"] == "complete" and snapshot.get("truncated") is False:
            return snapshot
    except Exception:
        pass  # Optional metadata must not prevent an authorized extension load.
    return None


def _spur_record_function_ownership(db, extension, before):
    """Retain registrations observed around a verified load on this connection."""
    if before is None:
        return
    try:
        if before["status"] != "complete" or before.get("truncated") is not False:
            return
        after = _spur_snapshot_function_registration(db)
        if after is None:
            return
        def key(row):
            return (*_spur_function_identity(row), row.get("return_type"), row["function_oid"])
        previous = {key(row) for row in before["functions"]}
        registrations = globals().setdefault("_SPUR_DUCKDB_FUNCTION_OWNERSHIP", {})
        retained = registrations.get(id(db))
        owners = dict(retained[1]) if retained is not None and retained[0] is db else {}
        additions = [key(row) for row in after["functions"]
                     if row["function_oid"] is not None and key(row) not in previous]
        for identity in additions:
            owners[identity] = set(owners.get(identity, ())) | {extension}
        registrations[id(db)] = (db, owners)
    except Exception:
        pass  # Incomplete/failed evidence stays unattributed.


class _SpurDuckDBRuntime:
    BASELINE = ("json", "parquet", "httpfs", "spur_rest")
    REPOSITORIES = {
        # DuckDB's built-in installer handles HTTP without httpfs; HTTPS itself
        # requires httpfs. Use the official signed distribution bootstrap, never
        # temporarily re-enable autoload to download its own dependency.
        "httpfs": "http://extensions.duckdb.org",
        "spatial": "http://extensions.duckdb.org",
        "jev": "http://community-extensions.duckdb.org",
    }

    def __init__(self, db, platform):
        self.db, self.platform = db, platform
        self.entries = {}
        self.inventories = {}  # inert metadata; never activation authority
        self.verified = {}  # at most one private verified copy per curated entry
        self.loaded = set()  # artifact verification is not selected-connection load

    def _entry(self, name):
        return self.entries.setdefault(name, {
            "name": name, "installed": None,
            "desired_enabled": name in self.BASELINE, "loaded": None,
            "provider_health": "unknown", "restart_required": False,
            "activation_transport_restart": False,
            "stage": "idle", "reason": None, "policy": "unknown",
            "artifact_digest": None, "signature_verified": False,
        })

    def _observe(self, db, name):
        return db.sql(
            "SELECT loaded, installed, install_path, installed_from, install_mode "
            "FROM duckdb_extensions() WHERE extension_name = ?", params=[name]
        ).fetchone()

    def _refresh(self, entry):
        row = self._observe(self.db, entry["name"])
        entry["loaded"] = bool(row and row[0])
        entry["installed"] = bool(row and row[1])
        return row

    def _artifact(self, name, install, cancelled):
        # Verification happens in a separate database with signature and ABI
        # checks ON, even when the selected gateway connection allows unsigned
        # bundled code. A repository label or hash alone is not signing evidence.
        import duckdb
        import hashlib
        from pathlib import Path
        import shutil
        import tempfile
        directory = self.db.sql("SELECT current_setting('extension_directory')").fetchone()[0]
        config = {"allow_unsigned_extensions": "false",
                  "allow_community_extensions": "true" if name == "jev" else "false",
                  "allow_extensions_metadata_mismatch": "false",
                  "autoload_known_extensions": "false",
                  "autoinstall_known_extensions": "false"}
        config["home_directory"] = self.db.sql("SELECT current_setting('home_directory')").fetchone()[0]
        if directory:
            config["extension_directory"] = directory
        entry = self._entry(name)
        entry["signature_verified"] = False
        with duckdb.connect(config=config) as verifier:
            if (verifier.sql("SELECT version()").fetchone()[0] != "v1.5.5"
                    or verifier.sql("PRAGMA platform").fetchone()[0] != self.platform):
                raise ValueError("runtime_mismatch")
            entry["stage"] = "installing" if install else "verifying"
            row = self._observe(verifier, name)
            if install and (row is None or not row[1]):
                # Install only when absent. DuckDB rejects INSTALL across HTTP/
                # HTTPS origin spellings even for the same official repository.
                # Cached artifacts follow the same origin/signature/ABI checks
                # below; never FORCE/UPDATE or silently replace rejected bytes.
                verifier.install_extension(name, repository_url=self.REPOSITORIES[name])
                row = self._observe(verifier, name)
            if row is None or not row[1]:
                raise ValueError("not_installed")
            entry["installed"] = True
            if cancelled():
                raise InterruptedError()
            entry["stage"] = "verifying"
            repository = self.REPOSITORIES[name]
            aliases = (repository, repository.replace("http:", "https:"),
                       "community" if name == "jev" else "core")
            if row[3] not in aliases or row[4] != "REPOSITORY":
                raise ValueError("repository_mismatch")
            # Freeze the bytes in a private directory before the verifier loads
            # them. Load those same bytes into the selected connection, never a
            # mutable cache alias. Same-user/kernel arbitrary code is trusted.
            private = tempfile.TemporaryDirectory(prefix="spur-duckdb-verified-")
            try:
                path = Path(private.name) / (name + ".duckdb_extension")
                shutil.copyfile(row[2], path)
                path.chmod(0o400)
                digest = hashlib.sha256(path.read_bytes()).hexdigest()
                try:
                    before = _spur_function_catalog(verifier)
                except Exception:
                    before = None
                verifier.load_extension(str(path))  # engine checks signature/ABI
                if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
                    raise ValueError("artifact_changed")
                try:
                    after = _spur_function_catalog(verifier)
                    if before is None:
                        raise ValueError("inventory_failed")
                    previous_rows = {_spur_function_identity(f): f for f in before["functions"]}
                    functions = [f for f in after["functions"]
                                 if previous_rows.get(_spur_function_identity(f)) != f]
                    complete = before["status"] == after["status"] == "complete"
                    inventory = {"status": "complete" if complete else "incomplete",
                                 "functions": functions, "truncated": not complete}
                except Exception:
                    inventory = {"status": "failed", "functions": [], "truncated": False}
                inventory.update(artifact_digest=digest, provenance="isolated_signed_verifier_load_delta")
                self.inventories[name] = inventory
            except BaseException:
                private.cleanup()
                raise
            previous = self.verified.get(name)
            if previous is not None:
                previous.cleanup()
            self.verified[name] = private
            return str(path), digest

    def operate(self, name, action, cancelled=lambda: False):
        """Explicit, nonsecret job; called only on the kernel shell queue.

        Cancellation is cooperative at stages. It never interrupts a user query;
        installed but not loaded artifacts remain available for an explicit retry.
        There is no UNLOAD in DuckDB: disable reports loaded + restart_required.
        """
        entry = self._entry(name)
        entry["stage"], entry["reason"] = "resolving", None
        if action == "disable" and name not in self.BASELINE:
            entry["desired_enabled"] = False
            if name == "jev":
                _spur_close_jev_channel()
        try:
            if name not in (*self.BASELINE, "spatial", "jev"):
                raise ValueError("unsupported_extension")
            if self.db is None:
                raise ValueError("no_connection")
            if self.db is not globals().get("_SPUR_DUCKDB_CONNECTION"):
                entry["policy"] = "unenforceable"
                raise ValueError("unmanaged_connection")
            if (self.db.sql("SELECT version()").fetchone()[0] != "v1.5.5"
                    or self.db.sql("PRAGMA platform").fetchone()[0] != self.platform):
                raise ValueError("runtime_mismatch")
            if _spur_managed_connection(self.db) is None:
                entry["policy"] = "unenforceable"
                raise ValueError("policy_unenforceable")
            entry["policy"] = "enforced"
            self._refresh(entry)
            if action == "disable":
                if name in self.BASELINE:
                    raise ValueError("required_baseline")
                entry["desired_enabled"] = False
                entry["restart_required"] = entry["loaded"] or name == "jev"
                entry["activation_transport_restart"] = name == "jev"
                entry["reason"] = "restart_required" if entry["restart_required"] else None
                entry["stage"] = "succeeded"
                return dict(entry)
            if cancelled():
                raise InterruptedError()
            if action == "invalidate_credential_channel" and name == "jev":
                _spur_close_jev_channel()
                entry["restart_required"] = True
                entry["activation_transport_restart"] = True
                entry["reason"] = "restart_required"
            elif action == "enable":
                entry["desired_enabled"] = True
                if name == "jev" and not self._channel_ready():
                    entry["restart_required"] = True
                    entry["activation_transport_restart"] = True
                    entry["reason"] = self._channel_reason()
            elif action in ("install", "revalidate", "load", "prepare_verification"):
                if action == "prepare_verification" and (name not in self.REPOSITORIES or not entry["installed"]
                        or not entry["signature_verified"] or not entry["artifact_digest"]):
                    raise ValueError("not_installed")
                if action == "load" and not entry["desired_enabled"]:
                    raise ValueError("setup_required_without_autoload")
                if name in ("json", "parquet"):
                    row = self._observe(self.db, name)
                    if not row or row[4] != "STATICALLY_LINKED" or not row[0]:
                        raise ValueError("baseline_unavailable")
                elif name == "spur_rest":
                    # Owned by the bundled artifact installer/bootstrap. Never
                    # weaken signature policy or download a substitute here.
                    if not entry["loaded"]:
                        raise ValueError("baseline_unavailable")
                else:
                    if entry["loaded"] and name == "httpfs" and name not in self.loaded:
                        # Connection setup may have loaded the same signed baseline
                        # first. Accept only its exact-connection frozen-byte proof.
                        from pathlib import Path
                        import hashlib
                        proof = globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {}).get((id(self.db), name))
                        if proof and proof[0] is self.db and proof[1] == "core":
                            try:
                                digest = hashlib.sha256((Path(proof[2].name) / "httpfs.duckdb_extension").read_bytes()).digest()
                                if digest == proof[3]:
                                    self.verified[name] = proof[2]
                                    self.loaded.add(name)
                                    entry.update(signature_verified=True, artifact_digest=digest.hex(),
                                                 restart_required=False, activation_transport_restart=False)
                            except OSError:
                                pass
                    if entry["loaded"] and name not in self.loaded:
                        entry["signature_verified"] = False
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = False
                        raise ValueError("preexisting_load_unverified")
                    if entry["loaded"] and action in ("load", "prepare_verification"):
                        # Explicit retry may reacquire discovery in the same isolated
                        # signature verifier; it must not change selected enablement.
                        if action == "prepare_verification" and self.inventories.get(name, {}).get("status") != "complete":
                            loaded_digest = entry["artifact_digest"]
                            _, refreshed_digest = self._artifact(name, False, cancelled)
                            if refreshed_digest != loaded_digest:
                                entry["restart_required"] = True
                                entry["activation_transport_restart"] = False
                                raise ValueError("artifact_changed")
                            entry["signature_verified"] = True
                        if name == "jev" and action == "load" and not self._channel_ready():
                            entry["restart_required"] = True
                            entry["activation_transport_restart"] = True
                            entry["reason"] = self._channel_reason()
                        if entry["restart_required"] and entry["activation_transport_restart"]:
                            entry["reason"] = self._channel_reason()
                        entry["stage"] = "succeeded"
                        return dict(entry)
                    if entry["loaded"]:
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = False
                        raise ValueError("restart_required")
                    path, digest = self._artifact(name, action == "install", cancelled)
                    entry["installed"] = True
                    entry["artifact_digest"] = digest
                    entry["signature_verified"] = True
                    if cancelled():
                        raise InterruptedError()
                    if action in ("load", "prepare_verification") and not entry["loaded"]:
                        entry["stage"] = "loading"
                        before = _spur_snapshot_function_registration(self.db)
                        self.db.load_extension(path)
                        self._refresh(entry)
                        if not entry["loaded"]:
                            raise ValueError("load_failed")
                        self.loaded.add(name)
                        _spur_record_function_ownership(self.db, name, before)
                    if name == "jev" and action != "prepare_verification" and not self._channel_ready():
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = True
                        entry["reason"] = self._channel_reason()
            elif action != "inspect":
                raise ValueError("unsupported_operation")
            if entry["restart_required"] and entry["activation_transport_restart"]:
                entry["reason"] = self._channel_reason()
            entry["stage"] = "succeeded"
        except (KeyboardInterrupt, InterruptedError):
            entry["reason"] = "cancelled"
        except ValueError as error:
            # Only our fixed reasons are exposed; never native exception text.
            allowed = {"runtime_mismatch", "unsupported_extension", "unmanaged_connection",
                       "required_baseline", "setup_required_without_autoload", "not_installed",
                       "repository_mismatch", "artifact_changed", "baseline_unavailable",
                       "preexisting_load_unverified", "load_failed", "unsupported_operation", "no_connection",
                       "policy_unenforceable", "restart_required"}
            entry["reason"] = str(error) if str(error) in allowed else "verification_failed"
        except Exception:
            entry["reason"] = {"installing": "install_failed", "loading": "load_failed",
                               "verifying": "verification_failed"}.get(entry["stage"], "policy_failed")
        return dict(entry)

    @staticmethod
    def _channel_ready():
        import os
        if os.name != "posix":
            return False
        try:
            channel = _spur_jev_channel()
            return channel is not None and channel.available()
        except Exception:
            return False

    @staticmethod
    def _channel_reason():
        import os
        return "restart_required" if os.name == "posix" else "credential_transport_unavailable"

    def close(self):
        """Release verified temporary copies after closing this connection."""
        for directory in self.verified.values():
            directory.cleanup()
        self.verified.clear()


def _spur_duckdb_job(platform, name, action, cancel_path=None):
    """Entry point for typed host jobs; no caller SQL and no credential payload."""
    from pathlib import Path
    global _SPUR_DUCKDB_RUNTIME
    db = globals().get("_SPUR_DUCKDB_CONNECTION")
    current = globals().get("_SPUR_DUCKDB_RUNTIME")
    if current is None or current.db is not db or current.platform != platform:
        current = _SPUR_DUCKDB_RUNTIME = _SpurDuckDBRuntime(db, platform)
    return current.operate(name, action, lambda: bool(cancel_path and Path(cancel_path).exists()))


def _spur_duckdb_baseline():
    """Load approved cached baseline only; missing artifacts need explicit setup.

    Used by both root bootstrap and the library's shared SQL session helper.
    The bundled gateway loader stays with its existing host-owned installer.
    """
    db = globals().get("_SPUR_DUCKDB_CONNECTION")
    if db is None or db.sql("SELECT version()").fetchone()[0] != "v1.5.5":
        return
    platform = db.sql("PRAGMA platform").fetchone()[0]
    current = globals().get("_SPUR_DUCKDB_RUNTIME")
    if current is not None and current.db is not db:
        current = None
    for name in ("json", "parquet", "httpfs"):
        if current is None or not current._entry(name)["loaded"]:
            _spur_duckdb_job(platform, name, "load")


class _SpurJevChannel:
    def __init__(self, descriptor):
        import socket
        self.socket = socket.socket(fileno=int(descriptor))
        self.socket.set_inheritable(False)
        self.lease = self._frame()

    def _read(self, count):
        chunks = bytearray()
        while len(chunks) < count:
            chunk = self.socket.recv(count - len(chunks))
            if not chunk:
                raise RuntimeError("Jev private credential channel closed")
            chunks.extend(chunk)
        return bytes(chunks)

    def _frame(self):
        return self._read(int.from_bytes(self._read(4), "big"))

    def _authorize(self, resolve):
        self.socket.sendall(len(self.lease).to_bytes(4, "big") + self.lease + bytes([resolve]))
        status = self._read(1)
        if status == b"\x02":
            return False
        if status != b"\x01":
            raise RuntimeError("denied")
        return True

    def available(self):
        try:
            return self._authorize(False)
        except Exception:
            self.socket.close()
            raise RuntimeError("Jev credential request denied or unavailable. Verify and enable the current managed local kernel in Settings.") from None

    def request(self):
        try:
            if not self._authorize(True):
                return None
            token = self._frame().decode("ascii")
            if not token or any(not 33 <= ord(c) <= 126 for c in token):
                raise ValueError("invalid credential")
            return token
        except Exception:
            self.socket.close()
            raise RuntimeError("Jev credential request denied or unavailable. Verify and enable the current managed local kernel in Settings.") from None


# Repeated generated SQL preambles preserve the captured channel. Failed
# attachment remains failed until kernel restart; never retry a legacy file.
if "_SPUR_JEV_BROKER" not in globals():
    _SPUR_JEV_BROKER = None
    _SPUR_JEV_BROKER_FAILED = False


def _spur_jev_channel():
    import os
    global _SPUR_JEV_BROKER, _SPUR_JEV_BROKER_FAILED
    if _SPUR_JEV_BROKER_FAILED:
        raise RuntimeError("Jev private credential channel is unavailable. Restart the managed local kernel.")
    descriptor = os.environ.pop("SPUR_JEV_BROKER_FD", None)
    if descriptor is not None:
        try:
            if _SPUR_JEV_BROKER is not None:
                raise RuntimeError("duplicate channel")
            _SPUR_JEV_BROKER = _SpurJevChannel(descriptor)
        except Exception:
            _SPUR_JEV_BROKER_FAILED = True
            raise RuntimeError("Jev private credential channel is unavailable. Restart the managed local kernel.") from None
    return _SPUR_JEV_BROKER


def _spur_close_jev_channel():
    """Disable/revoke/replacement is terminal for this inherited descriptor."""
    import os
    global _SPUR_JEV_BROKER_FAILED
    _SPUR_JEV_BROKER_FAILED = True
    if _SPUR_JEV_BROKER is not None:
        _SPUR_JEV_BROKER.socket.close()
    descriptor = os.environ.pop("SPUR_JEV_BROKER_FD", None)
    if descriptor is not None:
        try:
            os.close(int(descriptor))
        except (ValueError, OSError):
            pass


class _SpurJevSession:
    def __init__(self, db):
        self.db = db
        self.token = None
        self.previous = None

    def __enter__(self):
        import os
        _spur_duckdb_baseline()
        runtime = globals().get("_SPUR_DUCKDB_RUNTIME")
        connection = _spur_managed_connection(self.db)
        if runtime is None or runtime.db is not connection:
            return self
        entry = runtime._entry("jev")
        if not (entry["desired_enabled"] and entry["loaded"] and entry["signature_verified"]
                and "jev" in runtime.loaded):
            return self
        try:
            channel = _spur_jev_channel()
            if channel is None or not channel.available():
                entry["restart_required"] = True
                entry["activation_transport_restart"] = True
                entry["reason"] = "restart_required"
                return self
        except Exception:
            # A dead/revoked channel cannot renew in-place. Do not block ordinary
            # SQL, read a legacy file or install/load anything as a fallback.
            entry["restart_required"] = True
            entry["activation_transport_restart"] = True
            entry["reason"] = "restart_required"
            return self
        endpoint = self.db.sql("SELECT current_setting('jev_api_url')").fetchone()[0]
        if endpoint != "https://api.typesafe.ai/v1/systemone":
            raise RuntimeError("Managed Jev credentials require the TypeSafe HTTPS endpoint. Reset jev_api_url before running this cell.")
        self.db.sql("RESET jev_api_key")
        # Resolve only after installation and exact endpoint validation. The
        # broker rechecks lease, enablement and generation on every request.
        try:
            token = channel.request()
        except Exception:
            entry["restart_required"] = True
            entry["reason"] = "restart_required"
            return self
        if token is None:  # disabled while setup was in flight
            return self
        self.previous = os.environ.get("TYPESAFE_API_KEY")
        self.token = token
        os.environ["TYPESAFE_API_KEY"] = token
        return self

    def __exit__(self, error_type, error, traceback):
        import os
        import traceback as traceback_module

        if self.token is None:
            return False
        token = self.token
        self.token = None
        if self.previous is None:
            os.environ.pop("TYPESAFE_API_KEY", None)
        else:
            os.environ["TYPESAFE_API_KEY"] = self.previous
        self.previous = None
        # A remote API can echo a header in an error. Suppress the original
        # traceback chain and remove the exact managed token from the diagnostic.
        if error is not None and token in "".join(traceback_module.format_exception(error_type, error, traceback)):
            message = str(error).replace(token, "[redacted]")
            token = None
            error = None
            traceback = None
            raise RuntimeError(message) from None
        return False

def _spur_restore_catalog_extensions(connection, requirements):
    """Restore optional membership without making unrelated SQL depend on its success."""
    import sys
    failures = globals().setdefault("_SPUR_CATALOG_EXTENSION_FAILURES", {})
    errors = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        key = (id(connection), name, source)
        if _spur_catalog_extensions_ready(connection, [item]):
            failures.pop(key, None)
            continue
        previous = failures.get(key)
        if previous and previous[0] is connection:
            errors[name] = previous[1]
            continue
        try:
            _spur_prepare_catalog_extensions(connection, [item])
        except Exception as error:
            errors[name] = str(error)
            failures[key] = (connection, str(error))
            print(f"SPUR: optional extension {name} could not be restored: {error}. Retry or remove it in Extensions.", file=sys.stderr)
    return errors


def _spur_catalog_extensions_ready(connection, requirements):
    """Read current connection identity and proofs without installing or loading code."""
    import re
    rows = connection.sql(
        "SELECT extension_name, installed, loaded, install_mode, installed_from, "
        "install_path, aliases FROM duckdb_extensions()"
    ).fetchall()
    inventory = {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}
    proofs = globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            return False
        row = inventory.get(name)
        if not row or not row[2]:
            return False
        if row[3] == "STATICALLY_LINKED" and source == "core":
            continue
        proof = proofs.get((id(connection), row[0]))
        if not proof or proof[0] is not connection or proof[1] != source:
            return False
    return True


def _spur_prepare_connection_extensions(statements):
    """Use the same verifier for host-generated connection INSTALL/LOAD pairs."""
    import re
    requirements, loads = {}, set()
    for statement in statements:
        install = re.fullmatch(r"INSTALL ([a-z][a-z0-9_]*)(?: FROM (core|community))?", statement)
        load = re.fullmatch(r"LOAD ([a-z][a-z0-9_]*)", statement)
        if install:
            name, source = install.group(1), install.group(2) or "core"
            if name in requirements and requirements[name] != source:
                raise ValueError("Conflicting managed connection repositories")
            requirements[name] = source
        elif load:
            loads.add(load.group(1))
        else:
            raise ValueError("Invalid managed connection extension statement")
    if loads != set(requirements):
        raise ValueError("Managed connection extension installation and load must match")
    connection = globals().get("_SPUR_DUCKDB_CONNECTION")
    if connection is None:
        connection = duckdb.default_connection() if hasattr(duckdb, "default_connection") else duckdb
    _spur_prepare_catalog_extensions(connection, [
        {"packageName": name, "source": source} for name, source in requirements.items()
    ])


def _spur_prepare_catalog_extensions(connection, requirements):
    """Prepare signed packages on the notebook's shared DuckDB connection."""
    import duckdb as engine
    import hashlib
    from pathlib import Path
    import re
    import shutil
    import tempfile

    checked = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            raise ValueError("Invalid managed DuckDB extension identity")
        if name in checked and checked[name] != source:
            raise ValueError("One package cannot use two repositories in the same runtime")
        checked[name] = source
        # Explicit preparation is an intentional retry; ordinary SQL uses restore's memo.
        globals().get("_SPUR_CATALOG_EXTENSION_FAILURES", {}).pop((id(connection), name, source), None)

    def inventory(db):
        rows = db.sql(
            "SELECT extension_name, installed, loaded, install_mode, installed_from, "
            "install_path, aliases FROM duckdb_extensions()"
        ).fetchall()
        return {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}

    def matches(row, source):
        origin = row[4] or ""
        repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
        return origin == source or origin.rstrip("/") in (
            "http://" + repository, "https://" + repository)

    # Proofs belong to this exact connection, surviving redefinition on SQL replay.
    # Retain private immutable copies for the lifetime of the kernel.
    proofs = globals().setdefault("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for name, source in checked.items():
        row = inventory(connection).get(name)
        if row and row[3] == "STATICALLY_LINKED" and source == "core":
            if not row[2]:
                connection.load_extension(name)
            continue
        proof = proofs.get((id(connection), row[0] if row else name))
        if row and row[2]:
            # The required HTTPFS baseline has an existing signed verifier and
            # frozen copy. Share that exact proof, never a bare loaded flag or
            # repository label. This also supports already-running baseline code.
            if proof is None and row[0] == "httpfs" and source == "core":
                runtime = globals().get("_SPUR_DUCKDB_RUNTIME")
                if runtime is not None and runtime.db is connection and "httpfs" in runtime.loaded:
                    baseline = runtime._entry("httpfs")
                    private = runtime.verified.get("httpfs")
                    if baseline.get("signature_verified") is True and baseline.get("loaded") is True and private is not None:
                        try:
                            digest = hashlib.sha256((Path(private.name) / "httpfs.duckdb_extension").read_bytes()).digest()
                            if digest.hex() == baseline.get("artifact_digest"):
                                proof = (connection, source, private, digest)
                                proofs[(id(connection), row[0])] = proof
                        except OSError:
                            pass
            if proof and proof[0] is connection and proof[1] == source:
                continue
            raise RuntimeError(f"{name} is already loaded without this setup's verified artifact; restart the kernel and retry Install in notebook")

        # The shared connection permits the app's bundled unsigned gateway.
        # Always verify third-party bytes in an isolated engine with signing ON.
        config = {
            "allow_unsigned_extensions": "false",
            "allow_extensions_metadata_mismatch": "false",
            "allow_community_extensions": "true" if source == "community" else "false",
            "autoload_known_extensions": "false",
            "autoinstall_known_extensions": "false",
            "home_directory": connection.sql("SELECT current_setting('home_directory')").fetchone()[0],
        }
        directory = connection.sql("SELECT current_setting('extension_directory')").fetchone()[0]
        if directory:
            config["extension_directory"] = directory
        private = None
        try:
            with engine.connect(config=config) as verifier:
                for query in ("SELECT version()", "PRAGMA platform"):
                    if verifier.sql(query).fetchone() != connection.sql(query).fetchone():
                        raise RuntimeError("DuckDB runtime changed during extension verification")
                artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
                    verifier.install_extension(name, repository_url="http://" + repository)
                    artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    raise RuntimeError(f"{name} installation is unavailable for this DuckDB runtime")
                if artifact[3] != "REPOSITORY" or not matches(artifact, source):
                    raise RuntimeError(f"{name} is cached from another repository; resolve its source before retrying")
                private = tempfile.TemporaryDirectory(prefix="spur-duckdb-catalog-")
                path = Path(private.name) / (artifact[0] + ".duckdb_extension")
                shutil.copyfile(artifact[5], path)
                path.chmod(0o400)
                digest = hashlib.sha256(path.read_bytes()).digest()
                verifier.load_extension(str(path))  # signature and ABI checked by DuckDB
                if hashlib.sha256(path.read_bytes()).digest() != digest:
                    raise RuntimeError("Extension artifact changed during verification")
            snapshot = globals().get("_spur_snapshot_function_registration")
            before = snapshot(connection) if callable(snapshot) else None
            connection.load_extension(str(path))  # load exactly the verified bytes
            observed = inventory(connection).get(name)
            if not observed or not observed[2]:
                raise RuntimeError(f"{name} did not load into the notebook connection")
            proofs[(id(connection), observed[0])] = (connection, source, private, digest)
            record = globals().get("_spur_record_function_ownership")
            if callable(record):
                record(connection, observed[0], before)
            private = None
        finally:
            if private is not None:
                private.cleanup()


_spur_duckdb_baseline()
def _spur_restore_catalog_extensions(connection, requirements):
    """Restore optional membership without making unrelated SQL depend on its success."""
    import sys
    failures = globals().setdefault("_SPUR_CATALOG_EXTENSION_FAILURES", {})
    errors = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        key = (id(connection), name, source)
        if _spur_catalog_extensions_ready(connection, [item]):
            failures.pop(key, None)
            continue
        previous = failures.get(key)
        if previous and previous[0] is connection:
            errors[name] = previous[1]
            continue
        try:
            _spur_prepare_catalog_extensions(connection, [item])
        except Exception as error:
            errors[name] = str(error)
            failures[key] = (connection, str(error))
            print(f"SPUR: optional extension {name} could not be restored: {error}. Retry or remove it in Extensions.", file=sys.stderr)
    return errors


def _spur_catalog_extensions_ready(connection, requirements):
    """Read current connection identity and proofs without installing or loading code."""
    import re
    rows = connection.sql(
        "SELECT extension_name, installed, loaded, install_mode, installed_from, "
        "install_path, aliases FROM duckdb_extensions()"
    ).fetchall()
    inventory = {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}
    proofs = globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            return False
        row = inventory.get(name)
        if not row or not row[2]:
            return False
        if row[3] == "STATICALLY_LINKED" and source == "core":
            continue
        proof = proofs.get((id(connection), row[0]))
        if not proof or proof[0] is not connection or proof[1] != source:
            return False
    return True


def _spur_prepare_connection_extensions(statements):
    """Use the same verifier for host-generated connection INSTALL/LOAD pairs."""
    import re
    requirements, loads = {}, set()
    for statement in statements:
        install = re.fullmatch(r"INSTALL ([a-z][a-z0-9_]*)(?: FROM (core|community))?", statement)
        load = re.fullmatch(r"LOAD ([a-z][a-z0-9_]*)", statement)
        if install:
            name, source = install.group(1), install.group(2) or "core"
            if name in requirements and requirements[name] != source:
                raise ValueError("Conflicting managed connection repositories")
            requirements[name] = source
        elif load:
            loads.add(load.group(1))
        else:
            raise ValueError("Invalid managed connection extension statement")
    if loads != set(requirements):
        raise ValueError("Managed connection extension installation and load must match")
    connection = globals().get("_SPUR_DUCKDB_CONNECTION")
    if connection is None:
        connection = duckdb.default_connection() if hasattr(duckdb, "default_connection") else duckdb
    _spur_prepare_catalog_extensions(connection, [
        {"packageName": name, "source": source} for name, source in requirements.items()
    ])


def _spur_prepare_catalog_extensions(connection, requirements):
    """Prepare signed packages on the notebook's shared DuckDB connection."""
    import duckdb as engine
    import hashlib
    from pathlib import Path
    import re
    import shutil
    import tempfile

    checked = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            raise ValueError("Invalid managed DuckDB extension identity")
        if name in checked and checked[name] != source:
            raise ValueError("One package cannot use two repositories in the same runtime")
        checked[name] = source
        # Explicit preparation is an intentional retry; ordinary SQL uses restore's memo.
        globals().get("_SPUR_CATALOG_EXTENSION_FAILURES", {}).pop((id(connection), name, source), None)

    def inventory(db):
        rows = db.sql(
            "SELECT extension_name, installed, loaded, install_mode, installed_from, "
            "install_path, aliases FROM duckdb_extensions()"
        ).fetchall()
        return {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}

    def matches(row, source):
        origin = row[4] or ""
        repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
        return origin == source or origin.rstrip("/") in (
            "http://" + repository, "https://" + repository)

    # Proofs belong to this exact connection, surviving redefinition on SQL replay.
    # Retain private immutable copies for the lifetime of the kernel.
    proofs = globals().setdefault("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for name, source in checked.items():
        row = inventory(connection).get(name)
        if row and row[3] == "STATICALLY_LINKED" and source == "core":
            if not row[2]:
                connection.load_extension(name)
            continue
        proof = proofs.get((id(connection), row[0] if row else name))
        if row and row[2]:
            # The required HTTPFS baseline has an existing signed verifier and
            # frozen copy. Share that exact proof, never a bare loaded flag or
            # repository label. This also supports already-running baseline code.
            if proof is None and row[0] == "httpfs" and source == "core":
                runtime = globals().get("_SPUR_DUCKDB_RUNTIME")
                if runtime is not None and runtime.db is connection and "httpfs" in runtime.loaded:
                    baseline = runtime._entry("httpfs")
                    private = runtime.verified.get("httpfs")
                    if baseline.get("signature_verified") is True and baseline.get("loaded") is True and private is not None:
                        try:
                            digest = hashlib.sha256((Path(private.name) / "httpfs.duckdb_extension").read_bytes()).digest()
                            if digest.hex() == baseline.get("artifact_digest"):
                                proof = (connection, source, private, digest)
                                proofs[(id(connection), row[0])] = proof
                        except OSError:
                            pass
            if proof and proof[0] is connection and proof[1] == source:
                continue
            raise RuntimeError(f"{name} is already loaded without this setup's verified artifact; restart the kernel and retry Install in notebook")

        # The shared connection permits the app's bundled unsigned gateway.
        # Always verify third-party bytes in an isolated engine with signing ON.
        config = {
            "allow_unsigned_extensions": "false",
            "allow_extensions_metadata_mismatch": "false",
            "allow_community_extensions": "true" if source == "community" else "false",
            "autoload_known_extensions": "false",
            "autoinstall_known_extensions": "false",
            "home_directory": connection.sql("SELECT current_setting('home_directory')").fetchone()[0],
        }
        directory = connection.sql("SELECT current_setting('extension_directory')").fetchone()[0]
        if directory:
            config["extension_directory"] = directory
        private = None
        try:
            with engine.connect(config=config) as verifier:
                for query in ("SELECT version()", "PRAGMA platform"):
                    if verifier.sql(query).fetchone() != connection.sql(query).fetchone():
                        raise RuntimeError("DuckDB runtime changed during extension verification")
                artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
                    verifier.install_extension(name, repository_url="http://" + repository)
                    artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    raise RuntimeError(f"{name} installation is unavailable for this DuckDB runtime")
                if artifact[3] != "REPOSITORY" or not matches(artifact, source):
                    raise RuntimeError(f"{name} is cached from another repository; resolve its source before retrying")
                private = tempfile.TemporaryDirectory(prefix="spur-duckdb-catalog-")
                path = Path(private.name) / (artifact[0] + ".duckdb_extension")
                shutil.copyfile(artifact[5], path)
                path.chmod(0o400)
                digest = hashlib.sha256(path.read_bytes()).digest()
                verifier.load_extension(str(path))  # signature and ABI checked by DuckDB
                if hashlib.sha256(path.read_bytes()).digest() != digest:
                    raise RuntimeError("Extension artifact changed during verification")
            snapshot = globals().get("_spur_snapshot_function_registration")
            before = snapshot(connection) if callable(snapshot) else None
            connection.load_extension(str(path))  # load exactly the verified bytes
            observed = inventory(connection).get(name)
            if not observed or not observed[2]:
                raise RuntimeError(f"{name} did not load into the notebook connection")
            proofs[(id(connection), observed[0])] = (connection, source, private, digest)
            record = globals().get("_spur_record_function_ownership")
            if callable(record):
                record(connection, observed[0], before)
            private = None
        finally:
            if private is not None:
                private.cleanup()

import json as _spur_extension_json
_spur_prepare_catalog_extensions(_SPUR_DUCKDB_CONNECTION, _spur_extension_json.loads("[{\"packageName\":\"duckrouting\",\"source\":\"community\"},{\"packageName\":\"shellfs\",\"source\":\"community\"}]"))


# Panic-path exploration: Spur graph + shellfs

This notebook explores [Finding panics with Call Graph Analyzer](https://tweedegolf.nl/en/blog/242/finding-panics-with-call-graph-analyzer/) using the Spur source graph. **This is a source-level experiment, not CGA's MIR analysis or a panic-freedom proof.**

Run the cells from top to bottom. The managed setup cell restores the signed `shellfs` and `duckrouting` extensions. The remaining cells require `rg` in the notebook kernel's PATH and the local repository/analyst paths shown in the SQL. Source scans are live; the graph database is attached read-only.

- `shellfs` streams a fixed, read-only `rg --json` command into DuckDB.
- The analyst index supplies candidate call edges and symbol ranges. The index used for this run reported indexed commit `46b85c5` while the working checkout was `ea25f4f`. Refresh the graph before drawing current coverage conclusions; restart the kernel or detach the existing read-only attachment to pick up a refreshed database.
- `duckrouting` computes directed paths over recorded source-level call edges, including heuristic dynamic edges. No recorded path means only that this graph did not find one.
- Text matches include tests and comments; unresolved or heuristic edges are not compiler-verified. The candidate query includes only `.expect` and `.unwrap` labels; it does not cover implicit checks, dependency internals, or all panic syntax.


In [4]:
SELECT extension_name, installed, loaded, install_mode FROM duckdb_extensions() WHERE extension_name IN ('shellfs', 'duckrouting') ORDER BY extension_name;

extension_name,installed,loaded,install_mode
duckrouting,True,True,REPOSITORY
shellfs,True,True,REPOSITORY


In [5]:
SELECT * FROM read_csv($$printf 'name,score\nprobe,7\n' |$$, header = true);

name,score
probe,7


In [7]:
ATTACH IF NOT EXISTS '/Users/kevintruong/project/SpurCode/.spur/analyst.duckdb' AS spur_graph (READ_ONLY); SELECT graph_content_hash, graph_index_version, extractor_version, complete FROM spur_graph._meta;

graph_content_hash,graph_index_version,extractor_version,complete
8599245794571718b3153894d70d281c091e04ef3300eff4d66c0f3eb9fe6d59,4,2026-08-30-html-css-extraction-v1,True


In [8]:
SELECT * FROM read_csv($$command -v rg || printf 'RG_NOT_IN_PATH\n' |$$, columns = {'path': 'VARCHAR'}, header = false);

path
/opt/homebrew/bin/rg


In [9]:
SELECT table_name, column_name, data_type FROM information_schema.columns WHERE table_catalog = 'spur_graph' AND table_name IN ('nodes','edges','edges_unresolved') ORDER BY table_name, ordinal_position;

table_name,column_name,data_type
edges,source_stable_id,VARCHAR
edges,target_stable_id,VARCHAR
edges,src_id,BIGINT
edges,dst_id,BIGINT
edges,target_label,VARCHAR
edges,relation,VARCHAR
edges,confidence,VARCHAR
edges,confidence_score,FLOAT
edges,edge_kind,VARCHAR
edges,bind_method,VARCHAR


In [10]:
SELECT node_id, stable_symbol_id, qualified_name, file_path, line_start FROM spur_graph.nodes WHERE stable_symbol_id IN ('63f6898cbb3fcfdd', '5fb06d53a2477ff7', '10dad659a68a8b0f') ORDER BY file_path, line_start;

node_id,stable_symbol_id,qualified_name,file_path,line_start
36667,63f6898cbb3fcfdd,build_with_temporal_jobs_override,crates/spur-cli/src/commands/graph.rs,108
35096,5fb06d53a2477ff7,run_full_walk_into,crates/spur-graph/src/git_walk.rs,910
6133,10dad659a68a8b0f,run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,920


In [25]:
CREATE OR REPLACE TEMP TABLE panic_source_hits AS
SELECT
  replace(json_extract_string(data, '$.path.text'), '/Users/kevintruong/project/SpurCode/', '') AS file_path,
  try_cast(json_extract_string(data, '$.line_number') AS INTEGER) AS line_number,
  trim(json_extract_string(data, '$.lines.text')) AS source_line
FROM read_json(
  $$rg --json --glob '*.rs' -e '\.(unwrap|expect)\s*\(' -e '\b(panic|unreachable|todo|unimplemented)!\s*[\(\{\[]' /Users/kevintruong/project/SpurCode/crates/spur-graph/src |$$,
  format = 'newline_delimited',
  columns = {'type': 'VARCHAR', 'data': 'JSON'}
)
WHERE type = 'match';

SELECT count(*) AS matching_lines_including_tests_and_comments,
       count(DISTINCT file_path) AS files
FROM panic_source_hits;

matching_lines_including_tests_and_comments,files
1818,39


In [13]:
-- Node IDs are derived from the current graph snapshot; they can change on rebuild.
SET VARIABLE route_start = (SELECT node_id FROM spur_graph.nodes WHERE stable_symbol_id = '63f6898cbb3fcfdd');
SET VARIABLE route_target = (SELECT node_id FROM spur_graph.nodes WHERE stable_symbol_id = '10dad659a68a8b0f');

SELECT path.seq, path.node, n.qualified_name, n.file_path, path.agg_cost AS hops
FROM duckrouting_dijkstra(
  $$SELECT row_number() OVER ()::BIGINT AS id,
           src_id AS source, dst_id AS target,
           1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
    FROM spur_graph.edges
    WHERE relation = 'calls' AND edge_kind IN ('calls', 'calls_dyn')$$,
  getvariable('route_start'), getvariable('route_target')
) AS path
JOIN spur_graph.nodes n ON n.node_id = path.node
ORDER BY path.seq;

seq,node,qualified_name,file_path,hops
1,36667,build_with_temporal_jobs_override,crates/spur-cli/src/commands/graph.rs,0.0
2,35096,run_full_walk_into,crates/spur-graph/src/git_walk.rs,1.0
3,6133,run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,2.0


In [14]:
SELECT DISTINCT n.qualified_name, h.file_path, h.line_number, e.target_label,
       e.confidence AS graph_confidence, h.source_line
FROM panic_source_hits h
JOIN spur_graph.nodes n ON n.file_path = h.file_path
  AND h.line_number BETWEEN n.line_start AND n.line_end
JOIN spur_graph.edges_unresolved e ON e.source_stable_id = n.stable_symbol_id
WHERE n.stable_symbol_id = '10dad659a68a8b0f'
  AND e.target_label IN ('unwrap', 'expect')
  AND regexp_matches(h.source_line, '\.' || e.target_label || '\s*\(')
ORDER BY h.line_number;

qualified_name,file_path,line_number,target_label,graph_confidence,source_line
run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,958,expect,syntax_exact,"let base = base.expect(""incremental base checked above"");\n"


In [16]:
-- These are candidates, including possible test/comment matches, not proven panics.
SET VARIABLE route_start = (SELECT node_id FROM spur_graph.nodes WHERE stable_symbol_id = '63f6898cbb3fcfdd');

WITH reachable AS (
  SELECT node, agg_cost AS hops
  FROM duckrouting_driving_distance(
    $$SELECT row_number() OVER ()::BIGINT AS id,
             src_id AS source, dst_id AS target,
             1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
      FROM spur_graph.edges
      WHERE relation = 'calls' AND edge_kind IN ('calls', 'calls_dyn')$$,
    getvariable('route_start'), 3.0
  )
), candidates AS (
  SELECT r.hops, n.qualified_name, n.file_path, h.line_number, e.target_label,
         h.source_line,
         row_number() OVER (
           PARTITION BY h.file_path, h.line_number, e.target_label
           ORDER BY n.line_end - n.line_start
         ) AS smallest_symbol
  FROM reachable r
  JOIN spur_graph.nodes n ON n.node_id = r.node
  JOIN spur_graph.edges_unresolved e ON e.source_stable_id = n.stable_symbol_id
  JOIN panic_source_hits h ON h.file_path = n.file_path
       AND h.line_number BETWEEN n.line_start AND n.line_end
       AND regexp_matches(h.source_line, '\.' || e.target_label || '\s*\(')
  WHERE n.file_path LIKE 'crates/spur-graph/src/%'
    AND e.target_label IN ('unwrap', 'expect')
)
SELECT hops, qualified_name, file_path, line_number, target_label, source_line
FROM candidates
WHERE smallest_symbol = 1
ORDER BY hops, file_path, line_number
LIMIT 30;

hops,qualified_name,file_path,line_number,target_label,source_line
2.0,run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,958,expect,"let base = base.expect(""incremental base checked above"");\n"
2.0,write_edges,crates/spur-graph/src/store/parquet.rs,1720,expect,".expect(""resolved edge has target"")\n"
3.0,rebuild_from_buckets,crates/spur-graph/src/store/build.rs,2530,expect,".expect(""exhausted synthetic file NodeId space"");\n"
3.0,stable_file_id_from_path,crates/spur-graph/src/store/build.rs,2671,unwrap,u64::from_be_bytes(hasher.finalize()[..8].try_into().unwrap())\n


In [17]:
SET VARIABLE route_start = (SELECT node_id FROM spur_graph.nodes WHERE stable_symbol_id = '63f6898cbb3fcfdd');
SET VARIABLE route_target = (SELECT node_id FROM spur_graph.nodes WHERE stable_symbol_id = 'f7e5a680e7069785');
SELECT path.seq, n.qualified_name, n.file_path, path.agg_cost AS hops
FROM duckrouting_dijkstra(
  $$SELECT row_number() OVER ()::BIGINT AS id,
           src_id AS source, dst_id AS target,
           1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
    FROM spur_graph.edges
    WHERE relation = 'calls' AND edge_kind IN ('calls', 'calls_dyn')$$,
  getvariable('route_start'), getvariable('route_target')
) AS path
JOIN spur_graph.nodes n ON n.node_id = path.node
ORDER BY path.seq;

seq,qualified_name,file_path,hops
1,build_with_temporal_jobs_override,crates/spur-cli/src/commands/graph.rs,0.0
2,artifact_from_facts_incremental,crates/spur-graph/src/store/build.rs,1.0
3,compose_artifact,crates/spur-graph/src/store/build.rs,2.0
4,rebuild_from_buckets,crates/spur-graph/src/store/build.rs,3.0


In [18]:
-- The solver's witness, evaluated with wide SQL integers.
-- It says nothing about whether such an input can actually enter this function.
WITH model AS (
  SELECT CAST('18446744073709551614' AS HUGEINT) AS max_existing,
         true AS missing_file_id
)
SELECT max_existing, missing_file_id, max_existing + 1 AS next_synthetic,
       try_cast(max_existing + 2 AS UBIGINT) IS NULL AS next_checked_add_would_fail
FROM model;

max_existing,missing_file_id,next_synthetic,next_checked_add_would_fail
18446744073709551614,True,18446744073709551615,True


In [19]:
-- Exhaust the Boolean abstraction used by the previous solver check.
WITH inputs(fast_forward, base_present) AS (
  VALUES (false, false), (false, true), (true, false), (true, true)
), model AS (
  SELECT *, fast_forward AND base_present AS at_expect FROM inputs
)
SELECT count(*) AS assignments,
       count(*) FILTER (WHERE at_expect) AS assignments_reaching_expect,
       count(*) FILTER (WHERE at_expect AND NOT base_present) AS assignments_that_panic
FROM model;

assignments,assignments_reaching_expect,assignments_that_panic
4,1,0


## Interpretation and solver checks

The three-hop candidate query above identified four matching lines on the indexed call graph in this run. Those are **candidates**, not four demonstrated panics. The graph has unresolved call labels and source spans belong to symbols, so matching a current line to an older indexed function is a heuristic.

- `git_walk.rs:958`: at the `expect`, `use_incremental_base = fast_forward AND base_present`. A preflighted `solve_constraints` query found the branch satisfiable, and the branch plus `NOT base_present` **unsatisfiable**. The Boolean enumeration below independently evaluates all four input combinations. This result covers that manual abstraction, not other calls or paths.
- `store/build.rs:2530`: the 64-bit solver query returned **sat** for `max_existing = 0xfffffffffffffffe` and a file with no node ID. The next synthetic ID is `0xffffffffffffffff`, so incrementing it fails. The SQL witness below evaluates this arithmetic. This is a model-level boundary case; the experiment has not shown that a normal or accepted artifact can produce it.
- `store/parquet.rs:1720`: `edge_rows` creates resolved rows from `Some(target)`; proving every possible path into `write_edges` preserves that invariant needs a separate review.
- `store/build.rs:2671`: the source slices eight bytes from a SHA-256 digest before conversion. This suggests an invariant-based triage, but it was not separately solver-verified here.

Solver model for the 64-bit case (all constraints hard): `max_existing != all_ones`, `next_synthetic == bv_add(max_existing, 1)`, `missing_file_id == true`, and `next_synthetic == all_ones`. The typed request passed `solve_constraint_check` before `solve_constraints` returned the model. The solver operated separately through MCP; the SQL cell evaluates its recorded model rather than invoking Z3.

Useful next steps are a fresh graph rebuild, compiler-backed CGA output for the chosen binary, and a targeted test or invariant check for the boundary case. CGA and the source graph have different completeness and false-positive limits.


## Continuation: isolated current-worktree graph

The shared analyst index still served the earlier `859924...` graph. To continue without replacing that index, this section uses a separate graph built from the current, dirty worktree with `spur graph build --no-analyst --no-section-embeddings --no-code-symbol-embeddings`. Its content hash is `0855a502...`; its manifest has `indexed_commit_oid = null` because it is a filesystem snapshot, not a clean commit. The files are local under `.spur/graph/experiments/` and are not tracked in Git.

The source-OID check below verifies the specific files used by the findings. It does not certify every file in the snapshot or eliminate ambiguous, heuristic, unresolved, or conditional edges. Source-graph reachability remains a candidate, not proof of an execution path.


In [21]:
DESCRIBE SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/0855a502e1aabe9da17f2888253882bec044a67e8bf5b98e60324450b9e29e3d.parquet/nodes.parquet');

column_name,column_type,null,key,default,extra
stable_symbol_id,VARCHAR,YES,NaN,NaN,NaN
node_id,BIGINT,YES,NaN,NaN,NaN
file_path,VARCHAR,YES,NaN,NaN,NaN
byte_range_start,BIGINT,YES,NaN,NaN,NaN
byte_range_end,BIGINT,YES,NaN,NaN,NaN
line_start,INTEGER,YES,NaN,NaN,NaN
line_end,INTEGER,YES,NaN,NaN,NaN
entity_name,VARCHAR,YES,NaN,NaN,NaN
qualified_name,VARCHAR,YES,NaN,NaN,NaN
symbol_kind,VARCHAR,YES,NaN,NaN,NaN


In [6]:
CREATE OR REPLACE TEMP VIEW current_nodes AS SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/0855a502e1aabe9da17f2888253882bec044a67e8bf5b98e60324450b9e29e3d.parquet/nodes.parquet');
CREATE OR REPLACE TEMP VIEW current_edges AS SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/0855a502e1aabe9da17f2888253882bec044a67e8bf5b98e60324450b9e29e3d.parquet/edges.parquet');
CREATE OR REPLACE TEMP VIEW current_unresolved AS SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/0855a502e1aabe9da17f2888253882bec044a67e8bf5b98e60324450b9e29e3d.parquet/edges_unresolved.parquet');
CREATE OR REPLACE TEMP VIEW current_manifests AS SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/0855a502e1aabe9da17f2888253882bec044a67e8bf5b98e60324450b9e29e3d.parquet/file_manifests.parquet');
SELECT table_name, column_name, data_type FROM information_schema.columns
WHERE table_name IN ('current_nodes', 'current_edges', 'current_unresolved', 'current_manifests')
ORDER BY table_name, ordinal_position;

table_name,column_name,data_type
current_edges,source_stable_id,VARCHAR
current_edges,target_stable_id,VARCHAR
current_edges,src_id,BIGINT
current_edges,dst_id,BIGINT
current_edges,target_label,VARCHAR
current_edges,relation,VARCHAR
current_edges,confidence,VARCHAR
current_edges,confidence_score,FLOAT
current_edges,edge_kind,VARCHAR
current_edges,bind_method,VARCHAR


In [23]:
SELECT column_name, data_type FROM information_schema.columns WHERE table_name = 'current_manifests' ORDER BY ordinal_position;

column_name,data_type
stable_file_id,VARCHAR
path,VARCHAR
content_oid,VARCHAR
node_ids,BIGINT[]


In [34]:
-- Verify the six source files used by the sampled findings.
WITH live AS (
  SELECT * FROM read_csv(
    $$for f in crates/spur-cli/src/commands/graph.rs crates/spur-graph/src/git_walk.rs crates/spur-graph/src/store/build.rs crates/spur-graph/src/store/parquet.rs crates/spur-graph/src/validation.rs crates/spur-graph/src/extract/tree_sitter.rs; do printf '%s,' "$f"; git -C /Users/kevintruong/project/SpurCode hash-object --no-filters "$f"; done |$$,
    header = false, columns = {'path': 'VARCHAR', 'live_oid': 'VARCHAR'}
  )
)
SELECT live.path, live.live_oid, m.content_oid AS indexed_oid,
       live.live_oid = m.content_oid AS content_matches
FROM live
LEFT JOIN current_manifests m ON m.path = live.path
ORDER BY live.path;

path,live_oid,indexed_oid,content_matches
crates/spur-cli/src/commands/graph.rs,6e4f6ea108705455894cd320e0f045450c3a5042,6e4f6ea108705455894cd320e0f045450c3a5042,True
crates/spur-graph/src/extract/tree_sitter.rs,055dcf989b46a5de12e54774dcfbaf8a147fe1aa,055dcf989b46a5de12e54774dcfbaf8a147fe1aa,True
crates/spur-graph/src/git_walk.rs,aa96dae0a8e806b8fc3bb1adb33a655b98c9b229,aa96dae0a8e806b8fc3bb1adb33a655b98c9b229,True
crates/spur-graph/src/store/build.rs,4df2bddf13df7ac40c1f0e38df9ed7c6790523c8,4df2bddf13df7ac40c1f0e38df9ed7c6790523c8,True
crates/spur-graph/src/store/parquet.rs,d2d7a08b5bf0a491174fa1769f74f34c6008d194,d2d7a08b5bf0a491174fa1769f74f34c6008d194,True
crates/spur-graph/src/validation.rs,3ba906e69ca1b4691070453a6944a72b24a847a2,3ba906e69ca1b4691070453a6944a72b24a847a2,True


In [26]:
CREATE OR REPLACE TEMP TABLE current_call_edges AS
SELECT row_number() OVER ()::BIGINT AS id, src_id AS source, dst_id AS target,
       1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
FROM current_edges
WHERE relation = 'calls' AND edge_kind IN ('calls', 'calls_dyn')
  AND src_id IS NOT NULL AND dst_id IS NOT NULL;
SELECT count(*) AS directed_edges FROM current_call_edges;

directed_edges
40345


In [7]:
SET VARIABLE current_root = (
  SELECT node_id FROM current_nodes WHERE stable_symbol_id = '63f6898cbb3fcfdd'
);
-- The routing extension prepares its query on a separate connection, so it
-- reads the Parquet artifact directly instead of using a temporary table.
CREATE OR REPLACE TEMP TABLE current_reachable AS
SELECT node, agg_cost AS hops
FROM duckrouting_driving_distance(
  $$SELECT row_number() OVER ()::BIGINT AS id,
           src_id AS source, dst_id AS target,
           1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
    FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/0855a502e1aabe9da17f2888253882bec044a67e8bf5b98e60324450b9e29e3d.parquet/edges.parquet')
    WHERE relation = 'calls' AND edge_kind IN ('calls', 'calls_dyn')
      AND src_id IS NOT NULL AND dst_id IS NOT NULL$$,
  getvariable('current_root'), 5.0
);
SELECT hops, count(*) AS reachable_symbols
FROM current_reachable
GROUP BY hops ORDER BY hops;

hops,reachable_symbols
0.0,1
1.0,26
2.0,48
3.0,59
4.0,54
5.0,82


In [29]:
CREATE OR REPLACE TEMP TABLE current_candidates AS
WITH matched AS (
  SELECT r.hops, n.stable_symbol_id, n.qualified_name, n.file_path,
         h.line_number, e.target_label, h.source_line,
         row_number() OVER (
           PARTITION BY h.file_path, h.line_number, e.target_label
           ORDER BY n.line_end - n.line_start
         ) AS smallest_symbol
  FROM current_reachable r
  JOIN current_nodes n ON n.node_id = r.node
  JOIN current_unresolved e ON e.source_stable_id = n.stable_symbol_id
  JOIN panic_source_hits h ON h.file_path = n.file_path
       AND h.line_number BETWEEN n.line_start AND n.line_end
       AND regexp_matches(h.source_line, '\.' || e.target_label || '\s*\(')
  WHERE n.file_path LIKE 'crates/spur-graph/src/%'
    AND n.qualified_name NOT LIKE '%tests::%'
    AND e.relation = 'calls'
    AND e.target_label IN ('unwrap', 'expect')
)
SELECT hops, stable_symbol_id, qualified_name, file_path, line_number, target_label, source_line
FROM matched WHERE smallest_symbol = 1;
SELECT hops, qualified_name, file_path, line_number, target_label, source_line
FROM current_candidates ORDER BY hops, file_path, line_number LIMIT 50;

hops,qualified_name,file_path,line_number,target_label,source_line
2.0,run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,958,expect,"let base = base.expect(""incremental base checked above"");\n"
2.0,write_edges,crates/spur-graph/src/store/parquet.rs,1720,expect,".expect(""resolved edge has target"")\n"
3.0,rebuild_from_buckets,crates/spur-graph/src/store/build.rs,2530,expect,".expect(""exhausted synthetic file NodeId space"");\n"
3.0,stable_file_id_from_path,crates/spur-graph/src/store/build.rs,2671,unwrap,u64::from_be_bytes(hasher.finalize()[..8].try_into().unwrap())\n
4.0,impl ExternalImportRegistry::write_to_buckets,crates/spur-graph/src/store/build.rs,1460,expect,".expect(""exhausted synthetic external NodeId space"");\n"
4.0,writer_properties,crates/spur-graph/src/store/parquet.rs,2322,expect,".expect(""row-group row count fits u64"");\n"
4.0,compute_anchor_hash,crates/spur-graph/src/validation.rs,113,expect,".expect(""SHA-256 digest always has at least eight bytes""),\n"
5.0,resolve_bare_pending_edge,crates/spur-graph/src/extract/tree_sitter.rs,1076,expect,"fallback_edge.as_ref().expect(""fallback edge"")\n"


In [30]:
-- Compare source-text hits to graph labels. Labels are evidence of syntax,
-- not type-checked targets; a missing label cannot prove absence of a call.
CREATE OR REPLACE TEMP TABLE current_text_candidates AS
WITH matched AS (
  SELECT r.hops, n.stable_symbol_id, n.qualified_name, n.file_path,
         h.line_number, h.source_line,
         CASE
           WHEN regexp_matches(h.source_line, '\.(unwrap|expect)\s*\(')
             THEN regexp_extract(h.source_line, '\.(unwrap|expect)\s*\(', 1)
           WHEN regexp_matches(h.source_line, '\b(panic|unreachable|todo|unimplemented)!\s*[\(\{\[]')
             THEN regexp_extract(h.source_line, '\b(panic|unreachable|todo|unimplemented)!\s*[\(\{\[]', 1)
         END AS sink_label,
         row_number() OVER (
           PARTITION BY h.file_path, h.line_number
           ORDER BY n.line_end - n.line_start
         ) AS smallest_symbol
  FROM current_reachable r
  JOIN current_nodes n ON n.node_id = r.node
  JOIN panic_source_hits h ON h.file_path = n.file_path
       AND h.line_number BETWEEN n.line_start AND n.line_end
  WHERE n.file_path LIKE 'crates/spur-graph/src/%'
    AND n.qualified_name NOT LIKE '%tests::%'
    AND n.symbol_kind IN ('function', 'method')
)
SELECT * EXCLUDE (smallest_symbol) FROM matched
WHERE smallest_symbol = 1 AND sink_label IS NOT NULL;

WITH labels AS (
  SELECT source_stable_id, target_label, 'resolved' AS state FROM current_edges
  WHERE relation = 'calls'
  UNION ALL
  SELECT source_stable_id, target_label, 'unresolved' AS state FROM current_unresolved
  WHERE relation = 'calls'
)
SELECT t.hops, t.qualified_name, t.file_path, t.line_number, t.sink_label,
       coalesce(string_agg(DISTINCT l.state, ', '), 'no matching label') AS graph_label_state,
       t.source_line
FROM current_text_candidates t
LEFT JOIN labels l ON l.source_stable_id = t.stable_symbol_id AND l.target_label = t.sink_label
GROUP BY ALL
ORDER BY t.hops, t.file_path, t.line_number
LIMIT 50;

hops,qualified_name,file_path,line_number,sink_label,graph_label_state,source_line
2.0,run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,958,expect,unresolved,"let base = base.expect(""incremental base checked above"");\n"
2.0,write_edges,crates/spur-graph/src/store/parquet.rs,1720,expect,unresolved,".expect(""resolved edge has target"")\n"
3.0,rebuild_from_buckets,crates/spur-graph/src/store/build.rs,2530,expect,unresolved,".expect(""exhausted synthetic file NodeId space"");\n"
3.0,stable_file_id_from_path,crates/spur-graph/src/store/build.rs,2671,unwrap,unresolved,u64::from_be_bytes(hasher.finalize()[..8].try_into().unwrap())\n
4.0,impl ExternalImportRegistry::write_to_buckets,crates/spur-graph/src/store/build.rs,1460,expect,unresolved,".expect(""exhausted synthetic external NodeId space"");\n"
4.0,writer_properties,crates/spur-graph/src/store/parquet.rs,2322,expect,unresolved,".expect(""row-group row count fits u64"");\n"
4.0,compute_anchor_hash,crates/spur-graph/src/validation.rs,113,expect,unresolved,".expect(""SHA-256 digest always has at least eight bytes""),\n"
5.0,resolve_bare_pending_edge,crates/spur-graph/src/extract/tree_sitter.rs,1076,expect,unresolved,"fallback_edge.as_ref().expect(""fallback edge"")\n"
5.0,parse_raw_diff,crates/spur-graph/src/git_walk.rs,3412,unreachable,no matching label,"unreachable!(""kind=Renamed implies status_kind in [R,C]; see lines 1339-1340"")\n"


In [33]:
SET VARIABLE current_root = (SELECT node_id FROM current_nodes WHERE stable_symbol_id = '63f6898cbb3fcfdd');
SET VARIABLE macro_target = (SELECT node_id FROM current_nodes WHERE stable_symbol_id = 'ee814999c7a2927a');
WITH path AS (
  SELECT seq, node, agg_cost,
         lead(node) OVER (ORDER BY seq) AS next_node
  FROM duckrouting_dijkstra(
    $$SELECT row_number() OVER ()::BIGINT AS id, src_id AS source, dst_id AS target,
             1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
      FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/0855a502e1aabe9da17f2888253882bec044a67e8bf5b98e60324450b9e29e3d.parquet/edges.parquet')
      WHERE relation = 'calls' AND edge_kind IN ('calls', 'calls_dyn')$$,
    getvariable('current_root'), getvariable('macro_target')
  )
)
SELECT p.seq, n.qualified_name, n.file_path, p.agg_cost AS hops,
       string_agg(DISTINCT e.edge_kind || ':' || e.confidence, ', ') AS outgoing_edge_evidence
FROM path p
JOIN current_nodes n ON n.node_id = p.node
LEFT JOIN current_edges e ON e.src_id = p.node AND e.dst_id = p.next_node
  AND e.relation = 'calls' AND e.edge_kind IN ('calls', 'calls_dyn')
GROUP BY p.seq, n.qualified_name, n.file_path, p.agg_cost
ORDER BY p.seq;

seq,qualified_name,file_path,hops,outgoing_edge_evidence
1,build_with_temporal_jobs_override,crates/spur-cli/src/commands/graph.rs,0.0,calls:syntax_exact
2,run_full_walk_into,crates/spur-graph/src/git_walk.rs,1.0,calls:syntax_exact
3,run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,2.0,calls:syntax_exact
4,compute_commit_walk_result,crates/spur-graph/src/git_walk.rs,3.0,calls:syntax_exact
5,file_changes_for_commit,crates/spur-graph/src/git_walk.rs,4.0,calls:syntax_exact
6,parse_raw_diff,crates/spur-graph/src/git_walk.rs,5.0,NaN


In [32]:
-- Exhaust all 256 status bytes and both gitlink branches.
-- kind_renamed follows its construction in parse_raw_diff; the panic arm
-- requires renamed after the R/C guard fails.
WITH inputs AS (
  SELECT i AS status_byte, gitlink
  FROM range(0, 256) AS bytes(i)
  CROSS JOIN (VALUES (false), (true)) AS flags(gitlink)
), modeled AS (
  SELECT *,
         status_byte IN (67, 82) AS rename_or_copy,
         NOT gitlink AND status_byte IN (67, 82) AS kind_renamed
  FROM inputs
)
SELECT count(*) AS modeled_assignments,
       count(*) FILTER (WHERE kind_renamed) AS renamed_assignments,
       count(*) FILTER (WHERE kind_renamed AND NOT rename_or_copy) AS unreachable_arm_assignments
FROM modeled;

modeled_assignments,renamed_assignments,unreachable_arm_assignments
512,2,0


In [35]:
WITH outgoing AS (
  SELECT target_label, relation FROM current_edges
  WHERE source_stable_id = 'ee814999c7a2927a'
  UNION ALL
  SELECT target_label, relation FROM current_unresolved
  WHERE source_stable_id = 'ee814999c7a2927a'
)
SELECT count(*) AS total_recorded_outgoing_edges,
       count(*) FILTER (WHERE target_label ILIKE '%unreachable%') AS labels_mentioning_unreachable
FROM outgoing;

total_recorded_outgoing_edges,labels_mentioning_unreachable
25,0


In [36]:
-- Sample the modeled clamp; the solver result below covers the full modeled domain.
WITH inputs(row_count) AS (
  VALUES (0::HUGEINT), (1::HUGEINT), (16384::HUGEINT), (16385::HUGEINT),
         (CAST('18446744073709551615' AS HUGEINT))
), clamped AS (
  SELECT row_count, greatest(1::HUGEINT, least(row_count, 16384::HUGEINT)) AS clamped_count
  FROM inputs
)
SELECT row_count, clamped_count, try_cast(clamped_count AS UBIGINT) IS NOT NULL AS fits_u64
FROM clamped ORDER BY row_count;

row_count,clamped_count,fits_u64
0,1,True
1,1,True
16384,16384,True
16385,16384,True
18446744073709551615,16384,True


### Current-snapshot results

The directed graph reaches **270 indexed symbols including the seed** within five hops. Joining those functions to the live text scan yields eight `.expect` / `.unwrap` candidates and one `unreachable!` candidate. These counts are scoped to this root, radius, source patterns, and graph; they are not a panic count or a completeness claim.

| Candidate | Experiment result |
|---|---|
| `git_walk.rs:958` | Guarded `expect`: the earlier Boolean counterexample query is unsatisfiable under its manually encoded condition. |
| `store/parquet.rs:1720` | `edge_rows` constructs resolved rows only when the target is present; checking all paths that can create or change those rows remains separate work. |
| `store/build.rs:2530` | A 64-bit solver model reaches `checked_add` failure with an existing maximum of `u64::MAX - 1` and a missing file ID. Real artifact reachability is not established. |
| `store/build.rs:2671` | The conversion takes eight bytes from a fixed-width SHA-256 digest; source inspection suggests the unwrap is guarded by that type invariant. |
| `store/build.rs:1460` | A second synthetic-node allocator has the same extreme-value condition, when it must allocate a new external node. Its real artifact reachability is not established. |
| `store/parquet.rs:2322` | A solver model of `row_count.clamp(1, 16384)` found no result outside that interval, for all non-negative modeled integers. Values in the interval fit `u64`. |
| `validation.rs:113` | This also converts eight bytes of a SHA-256 digest; the source indicates the same fixed-length invariant. |
| `extract/tree_sitter.rs:1076` | `fallback_edge` is set to `Some(...)` immediately before the `expect` on this branch, assuming the preceding call returns normally. |
| `git_walk.rs:3412` | The snapshot has 25 outgoing edges for `parse_raw_diff`, but none has a target label containing `unreachable`. Source inspection and the solver check below show that its renamed branch requires status `R` or `C`. |

For the final row, the typed Z3 request represented an 8-bit status and a Boolean gitlink flag. It encoded `kind_renamed = NOT is_gitlink AND status IN {67,82}`; the renamed branch was satisfiable, while adding `status NOT IN {67,82}` was **unsatisfiable**. The SQL cell independently enumerates all 512 combinations and finds zero violating assignments. This establishes the supplied abstraction, not compiler-derived path semantics.

The targeted existing Rust test `parse_raw_diff_handles_copy_status_like_rename` could not be run: both configured AWS builder attempts failed to resolve the required AMI. The wrapper then began a local fallback; it was interrupted in accordance with the repository's remote-build policy. No Rust test pass is claimed.

The fresh graph is a local ignored artifact. To regenerate it, rerun the build command described above, select its new content-hash directory, and update the Parquet paths in this section. Compare source OIDs again before interpreting results.

The Rust edge query at `crates/spur-graph/queries/rust/spur-edges.scm` captures normal call expressions and call-shaped tokens inside macro bodies; it does not currently have a capture for the invoked macro name itself. A prototype can record macro-invocation names and source spans as **heuristic** sink evidence, with fixtures for `panic!`, `unreachable!`, assertions, qualified or shadowed macros, and calls inside macro arguments. This alone would not detect compiler-inserted checks or expand macros.


## Coverage-gap analysis and an effectiveness experiment

The observed gap is specific: the source graph can reach a function that contains a macro invocation, but it does not retain the invoked macro name as source evidence. The experiment below compares current graph output with a prototype AST collector on a controlled fixture set. It measures **macro-invocation capture**, separately from whether an invocation resolves to a standard panic macro or can actually panic.

The prototype and fixtures stay outside the production crate. Measurements from a finite fixture set establish behavior on that set only.

### Recommendation: retain macro-invocation evidence, then classify it

The first change should add a **versioned, per-invocation evidence sidecar** in `spur-graph`. Each row should retain the owner symbol, macro path, exact source span, source OID, and declared execution conditions. Panic-risk queries can join those rows to the existing call graph. This closes the observed source-evidence gap without resolving a bare macro name to an unrelated function symbol.

The notebook prototype provides bounded evidence for this direction:

| Measurement | Current query | Prototype | Meaning |
|---|---:|---:|---|
| Direct invocation names captured in 25 fixtures | 0 / 29 | 29 / 29 | Query-level syntax capture, not panic-path recall |
| False positive invocation sites in 3 negative controls | 0 | 0 | Comments, strings, and unused macro-definition tokens |
| Fixture context-label errors | — | 0 | Only the declared fixture cases |
| Existing ordinary/body-call positive controls | captured | retained as a separate query | The prototype adds evidence; it does not replace those captures |
| Real Rust files parsed / matching snapshot | — | 60 / 60 | Source hashes checked before graph joins |
| Real invocation spans independently validated | — | 4,158 | Byte offsets and line coordinates checked |
| Exact function-owner bindings | — | 4,100 / 4,100 | No ambiguous or unmapped function sites; 58 file-scope sites retained separately |
| Named macro sink candidates in the 270-symbol neighborhood | no matching labels for the six sites | 6 | Five debug assertions and one `unreachable!`, all unresolved bare names |

The corpus scan labels 1,968 of 2,007 named sink candidates as test-context invocations. The remaining 39 contain 11 debug-assertion invocations. These are **prototype context labels**, not compiler-evaluated `cfg` results. This shows that execution context and build settings must be retained alongside capture coverage; otherwise newly visible test/debug sites can overwhelm useful results.

The earlier finite solver check already found no counterexample to the local `unreachable!` invariant. Recovering its source evidence therefore does not turn it into a demonstrated bug. The useful outcome is a complete, reviewable candidate record with a separate semantic disposition.

The evaluation uses the repository's current Rust edge-query bytes (SHA-256 recorded below) and `tree-sitter-rust 0.24.2`. Native Python parsing runs in a subprocess with `py-tree-sitter 0.25.2`, after an initial 0.26.0 corpus run disconnected the notebook kernel. Production Rust extractor integration, full indexing overhead, and compiler-backed panic recall remain unmeasured.

Parser references: [Python binding documentation](https://tree-sitter.github.io/py-tree-sitter/), [official binding releases](https://github.com/tree-sitter/py-tree-sitter/releases), and [Rust grammar 0.24.2](https://pypi.org/project/tree-sitter-rust/0.24.2/).


In [3]:
# Native parsing stays in a child process. The first 0.26.0 corpus run
# disconnected this kernel; the isolated experiment pins 0.25.2 and validates
# byte-derived coordinates. This does not change the Rust crate's dependency.
import subprocess
import sys
from pathlib import Path
_coverage_packages = Path("/private/tmp/spur-panic-coverage-python-0252")
_coverage_packages.mkdir(parents=True, exist_ok=True)
if not (_coverage_packages / "tree_sitter").exists():
    _install = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir",
         "--disable-pip-version-check", "--only-binary=:all:",
         "--index-url", "https://pypi.org/simple", "--target", str(_coverage_packages),
         "tree-sitter==0.25.2", "tree-sitter-rust==0.24.2"],
        text=True, capture_output=True, timeout=60
    )
    print("pip_exit_code:", _install.returncode)
    if _install.returncode:
        print(_install.stderr[-2000:])
        raise RuntimeError("Research dependency installation failed")
print("isolated_package_directory:", _coverage_packages)
print("Native parser modules are not imported into the notebook kernel.")


pip_exit_code: 0
isolated_package_directory: /private/tmp/spur-panic-coverage-python-0252
Native parser modules are not imported into the notebook kernel.


In [18]:
_coverage_fixture_source = r'''
# Expected outputs are declared before the collector.
# Each tuple is (case, Rust source, expected direct macro sites, expected context).
coverage_cases = [
 ("direct", 'fn direct(){ panic!("x"); }', [("direct","panic")], "production"),
 ("qualified", 'fn qualified(){ std::panic!("x"); core::unreachable!(); }',
  [("qualified","std::panic"),("qualified","core::unreachable")], "production"),
 ("delimiters", 'fn delimiters(){ todo!{"x"}; unimplemented!["x"]; }',
  [("delimiters","todo"),("delimiters","unimplemented")], "production"),
 ("assertions", 'fn assertions(){ assert!(false); assert_eq!(1,2); assert_ne!(1,1); }',
  [("assertions","assert"),("assertions","assert_eq"),("assertions","assert_ne")], "production"),
 ("debug", 'fn debug(){ debug_assert!(false); debug_assert_eq!(1,2); debug_assert_ne!(1,1); }',
  [("debug","debug_assert"),("debug","debug_assert_eq"),("debug","debug_assert_ne")], "production"),
 ("raw_name", 'fn raw_name(){ r#panic!("x"); }', [("raw_name","panic")], "production"),
 ("spaced", 'fn spaced(){ std :: panic!("x"); }', [("spaced","std::panic")], "production"),
 ("shadowed", 'macro_rules! panic { () => { 0 } } fn shadowed(){ panic!(); }',
  [("shadowed","panic")], "production"),
 ("custom", 'fn custom(){ app::panic!("x"); }', [("custom","app::panic")], "production"),
 ("alias", 'use std::panic as explode; fn alias(){ explode!("x"); }',
  [("alias","explode")], "production"),
 ("test_module", '#[cfg(test)] mod tests { fn test_module(){ panic!("x"); } }',
  [("test_module","panic")], "test"),
 ("test_function", '#[test] fn test_function(){ panic!("x"); }',
  [("test_function","panic")], "test"),
 ("feature", '#[cfg(feature="demo")] fn feature(){ panic!("x"); }',
  [("feature","panic")], "conditional"),
 ("nested_tokens", 'fn nested_tokens(){ wrapper!(panic!("maybe")); }',
  [("nested_tokens","wrapper")], "production"),
 ("stringify", 'fn stringify(){ stringify!(panic!("inert tokens")); }',
  [("stringify","stringify")], "production"),
 ("nested_function", 'fn outer(){ fn inner(){ panic!("x"); } unreachable!(); }',
  [("inner","panic"),("outer","unreachable")], "production"),
 ("argument_call", 'fn argument_call(){ format!("{}", helper()); }',
  [("argument_call","format")], "production"),
 ("comments_strings", 'fn comments_strings(){ /* panic!("fake"); */ let s = r#"unreachable!()"#; }',
  [], "production"),
 ("definition_only", 'macro_rules! unused_macro { () => { panic!("not expanded") } }',
  [], "production"),
 ("ordinary_call", 'fn ordinary_call(){ helper(); }', [], "production"),
 ("async_test", '#[tokio::test(flavor="current_thread")] async fn async_test(){ panic!("x"); }',
  [("async_test","panic")], "test"),
 ("cfg_any", '#[cfg(any(test, feature="extra"))] fn cfg_any(){ panic!("x"); }',
  [("cfg_any","panic")], "conditional"),
 ("cfg_attr", '#[cfg_attr(feature="demo", test)] fn cfg_attr(){ panic!("x"); }',
  [("cfg_attr","panic")], "conditional"),
 ("commented_path", 'fn commented_path(){ std /* trivia */ :: panic!("x"); }',
  [("commented_path","std::panic")], "production"),
 ("absolute_path", 'fn absolute_path(){ ::core::panic!("x"); }',
  [("absolute_path","::core::panic")], "production"),
]
print("fixture_cases:", len(coverage_cases))
print("expected_direct_invocations:", sum(len(case[2]) for case in coverage_cases))
print("zero-invocation_controls:", sum(not case[2] for case in coverage_cases))
for _case, _source, _expected, _ctx in coverage_cases:
    _tree = _coverage_parser.parse(_source.encode())
    if _tree.root_node.has_error:
        raise AssertionError(f"Fixture has a parse error: {_case}")
print("all_fixture_sources_parse:", True)

'''
print("_coverage_fixture_source: saved for isolated execution")


_coverage_fixture_source: saved for isolated execution


In [19]:
_coverage_collector_source = r'''
import re
import hashlib
from collections import Counter
from bisect import bisect_right
from tree_sitter import Query, QueryCursor

_sink_names = {
    "panic", "unreachable", "todo", "unimplemented",
    "assert", "assert_eq", "assert_ne",
    "debug_assert", "debug_assert_eq", "debug_assert_ne",
}
def _syntax_text(source_bytes, node):
    return source_bytes[node.start_byte:node.end_byte].decode("utf-8")

def _normalize_macro_path(source_bytes, node):
    if node.type == "scoped_identifier":
        path = node.child_by_field_name("path")
        name = node.child_by_field_name("name")
        prefix = _normalize_macro_path(source_bytes, path) if path is not None else ""
        suffix = _normalize_macro_path(source_bytes, name) if name is not None else ""
        return prefix + "::" + suffix
    return _syntax_text(source_bytes, node).removeprefix("r#")

def _preceding_attributes(source_bytes, node):
    attributes = []
    previous = node.prev_named_sibling
    while previous is not None and previous.type == "attribute_item":
        attributes.append(_syntax_text(source_bytes, previous))
        previous = previous.prev_named_sibling
    return attributes

def collect_macro_evidence(source_text, file_path="<fixture>"):
    source_bytes = source_text.encode("utf-8")
    line_starts = [0] + [i + 1 for i, byte in enumerate(source_bytes) if byte == 10]
    tree = _coverage_parser.parse(source_bytes)
    if tree.root_node.has_error:
        raise ValueError(f"Parse error in {file_path}")
    definitions = []
    def find_definitions(node):
        if node.type == "macro_definition":
            name = node.child_by_field_name("name")
            if name is not None:
                definitions.append((node.parent.id, node.end_byte,
                                    _syntax_text(source_bytes, name).removeprefix("r#")))
            return
        for child in node.named_children:
            find_definitions(child)
    find_definitions(tree.root_node)
    evidence = []
    def visit(node, ancestors, attributes):
        # A definition's tokens have no execution caller until macro expansion.
        if node.type == "macro_definition":
            return
        attrs = attributes + _preceding_attributes(source_bytes, node)
        lineage = ancestors + [node]
        if node.type == "macro_invocation":
            macro = node.child_by_field_name("macro")
            if macro is not None:
                macro_path = _normalize_macro_path(source_bytes, macro)
                leaf = macro.child_by_field_name("name") or macro
                caller = next((a for a in reversed(ancestors) if a.type == "function_item"), None)
                caller_name_node = caller.child_by_field_name("name") if caller is not None else None
                caller_name = _syntax_text(source_bytes, caller_name_node) if caller_name_node is not None else "<file>"
                ancestor_ids = {a.id for a in ancestors}
                bare_name = macro_path.split("::")[-1]
                local_shadow = "::" not in macro_path and any(
                    scope in ancestor_ids and end <= node.start_byte and name == bare_name
                    for scope, end, name in definitions
                )
                compact_attrs = [re.sub(r"\s+", "", a) for a in attrs]
                is_test = any(
                    a == "#[cfg(test)]" or re.match(r"#\[(?:(?:tokio|async_std)::)?test(?:\(|\])", a) is not None
                    for a in compact_attrs
                )
                context = "test" if is_test else (
                    "conditional" if any("#[cfg" in a for a in compact_attrs) else "production"
                )
                if local_shadow:
                    classification = "local_macro_definition"
                elif macro_path.removeprefix("::") in {f"{root}::{name}" for root in ("core", "std") for name in _sink_names}:
                    classification = "explicit_standard_path_candidate"
                elif "::" not in macro_path and bare_name in _sink_names:
                    classification = "unresolved_bare_sink_candidate"
                elif bare_name in _sink_names:
                    classification = "other_qualified_macro"
                else:
                    classification = "other_macro_name"
                evidence.append({
                    "file_path": file_path, "caller": caller_name, "macro_path": macro_path,
                    "line": bisect_right(line_starts, node.start_byte),
                    "start_byte": node.start_byte, "end_byte": node.end_byte,
                    "name_start_byte": leaf.start_byte, "name_end_byte": leaf.end_byte,
                    "caller_start_byte": caller.start_byte if caller is not None else None,
                    "caller_end_byte": caller.end_byte if caller is not None else None,
                    "context": context, "attributes": sorted(set(attrs)),
                    "classification": classification,
                    "profile_conditional": bare_name.startswith("debug_assert"),
                    "origin": "macro_invocation_ast",
                })
        for child in node.named_children:
            visit(child, lineage, attrs)
    visit(tree.root_node, [], [])
    return tree, evidence

_query_path = Path("/Users/kevintruong/project/SpurCode/crates/spur-graph/queries/rust/spur-edges.scm")
_query_bytes = _query_path.read_bytes()
_baseline_query = Query(_coverage_language, _query_bytes.decode())
coverage_rows = []
coverage_case_metrics = []
coverage_failures = []
_ordinary_call_captured = False
_argument_call_captured = False
for case, source_text, expected, expected_context in coverage_cases:
    tree, observed = collect_macro_evidence(source_text, case)
    expected_counts = Counter(expected)
    observed_counts = Counter((r["caller"], r["macro_path"]) for r in observed)
    captures = QueryCursor(_baseline_query).captures(tree.root_node)
    baseline_names = captures.get("call.name", []) + captures.get("macro_call.name", [])
    baseline_spans = {(n.start_byte, n.end_byte) for n in baseline_names}
    baseline_detected = Counter(
        (r["caller"], r["macro_path"]) for r in observed
        if (r["name_start_byte"], r["name_end_byte"]) in baseline_spans
    )
    tp = sum((observed_counts & expected_counts).values())
    fp = sum((observed_counts - expected_counts).values())
    fn = sum((expected_counts - observed_counts).values())
    context_errors = sum(r["context"] != expected_context for r in observed)
    coverage_case_metrics.append({
        "case": case, "expected": sum(expected_counts.values()),
        "baseline_tp": sum((baseline_detected & expected_counts).values()),
        "prototype_tp": tp, "prototype_fp": fp, "prototype_fn": fn,
        "context_errors": context_errors,
    })
    if fp or fn or context_errors:
        coverage_failures.append((case, fp, fn, context_errors))
    for row in observed:
        row["case"] = case
    coverage_rows.extend(observed)
    if case == "ordinary_call":
        _ordinary_call_captured = any(_syntax_text(source_text.encode(), n) == "helper" for n in baseline_names)
    if case == "argument_call":
        _argument_call_captured = any(_syntax_text(source_text.encode(), n) == "helper" for n in baseline_names)
    print(case, "expected", sum(expected_counts.values()), "baseline_macro_sites", sum(baseline_detected.values()),
          "prototype_tp", tp, "fp", fp, "fn", fn, "context_errors", context_errors)

print("fixture_failures:", coverage_failures)
print("ordinary_call_preserved_by_baseline:", _ordinary_call_captured)
print("macro_argument_call_preserved_by_baseline:", _argument_call_captured)
print("baseline_query_sha256:", hashlib.sha256(_query_bytes).hexdigest())
assert not coverage_failures
assert _ordinary_call_captured and _argument_call_captured

'''
print("_coverage_collector_source: saved for isolated execution")


_coverage_collector_source: saved for isolated execution


In [20]:
# Run native parsing in an isolated child process and import only validated JSON.
import json
import subprocess
import sys
import tempfile
from pathlib import Path
_repo_root = Path("/Users/kevintruong/project/SpurCode")
_coverage_db = _SPUR_DUCKDB_CONNECTION
_header = ("import sys\nfrom pathlib import Path\n"
           + "sys.path.insert(0, " + repr(str(_coverage_packages)) + ")\n"
           + "import tree_sitter_rust\nfrom tree_sitter import Language, Parser\n"
           + "_coverage_language = Language(tree_sitter_rust.language())\n"
           + "_coverage_parser = Parser(_coverage_language)\n")
_scan_source = "import json\nimport importlib.metadata\n_repo = Path(\"/Users/kevintruong/project/SpurCode\")\n_live_rows, _file_rows, _parse_errors = [], [], []\nfor _file in sorted((_repo / \"crates/spur-graph/src\").rglob(\"*.rs\")):\n    _relative = _file.relative_to(_repo).as_posix()\n    _bytes = _file.read_bytes()\n    _oid = hashlib.sha1(b\"blob \" + str(len(_bytes)).encode() + b\"\\0\" + _bytes).hexdigest()\n    _file_rows.append({\"file_path\": _relative, \"source_oid\": _oid})\n    try:\n        _tree, _rows = collect_macro_evidence(_bytes.decode(\"utf-8\"), _relative)\n    except (ValueError, UnicodeDecodeError) as _error:\n        _parse_errors.append((_relative, str(_error)))\n        continue\n    for _row in _rows:\n        assert 0 <= _row[\"start_byte\"] < _row[\"end_byte\"] <= len(_bytes)\n        assert _row[\"line\"] == _bytes[:_row[\"start_byte\"]].count(bytes([10])) + 1\n        _row[\"source_oid\"] = _oid\n    _live_rows.extend(_rows)\n_result = {\n    \"runtime\": {\"tree_sitter\": importlib.metadata.version(\"tree-sitter\"),\n                \"tree_sitter_rust\": importlib.metadata.version(\"tree-sitter-rust\")},\n    \"fixture_metrics\": coverage_case_metrics,\n    \"fixture_evidence\": coverage_rows,\n    \"fixture_failures\": coverage_failures,\n    \"query_sha256\": hashlib.sha256(_query_bytes).hexdigest(),\n    \"ordinary_call_captured\": _ordinary_call_captured,\n    \"macro_argument_call_captured\": _argument_call_captured,\n    \"source_files\": _file_rows,\n    \"live_evidence\": _live_rows,\n    \"parse_errors\": _parse_errors,\n}\nPath(sys.argv[1]).write_text(json.dumps(_result), encoding=\"utf-8\")\n"
with tempfile.TemporaryDirectory(prefix="spur-macro-benchmark-", dir="/private/tmp") as _job:
    _script = Path(_job) / "benchmark.py"
    _output = Path(_job) / "result.json"
    _script.write_text(_header + _coverage_fixture_source + "\n" + _coverage_collector_source + "\n" + _scan_source)
    _process = subprocess.run([sys.executable, str(_script), str(_output)],
                              text=True, capture_output=True, timeout=60)
    print("parser_process_exit:", _process.returncode)
    if _process.returncode != 0:
        print(_process.stderr[-3000:])
        raise RuntimeError("Isolated parser did not finish; no partial results imported")
    _coverage_data = json.loads(_output.read_text())
print("parser_runtime:", _coverage_data["runtime"])
coverage_case_metrics = _coverage_data["fixture_metrics"]
coverage_rows = _coverage_data["fixture_evidence"]
live_macro_rows = _coverage_data["live_evidence"]
live_macro_parse_errors = _coverage_data["parse_errors"]
_snapshot_oids = dict(_coverage_db.execute("SELECT path, content_oid FROM current_manifests").fetchall())
live_file_rows = [(r["file_path"], r["source_oid"], _snapshot_oids.get(r["file_path"]),
                   r["source_oid"] == _snapshot_oids.get(r["file_path"]))
                  for r in _coverage_data["source_files"]]
for _row in live_macro_rows:
    _row["matches_snapshot"] = _row["source_oid"] == _snapshot_oids.get(_row["file_path"])
_coverage_db.execute("""CREATE OR REPLACE TEMP TABLE macro_fixture_metrics(
    case_name VARCHAR, expected BIGINT, baseline_tp BIGINT,
    prototype_tp BIGINT, prototype_fp BIGINT, prototype_fn BIGINT, context_errors BIGINT)""")
_coverage_db.executemany("INSERT INTO macro_fixture_metrics VALUES (?, ?, ?, ?, ?, ?, ?)", [
    (m["case"], m["expected"], m["baseline_tp"], m["prototype_tp"],
     m["prototype_fp"], m["prototype_fn"], m["context_errors"]) for m in coverage_case_metrics
])
_coverage_db.execute("""CREATE OR REPLACE TEMP TABLE macro_source_files(
    file_path VARCHAR, source_oid VARCHAR, snapshot_oid VARCHAR, matches_snapshot BOOLEAN)""")
_coverage_db.executemany("INSERT INTO macro_source_files VALUES (?, ?, ?, ?)", live_file_rows)
_coverage_db.execute("""CREATE OR REPLACE TEMP TABLE macro_live_evidence(
    file_path VARCHAR, caller VARCHAR, macro_path VARCHAR, line_number INTEGER,
    start_byte BIGINT, end_byte BIGINT, caller_start_byte BIGINT, caller_end_byte BIGINT,
    context VARCHAR, classification VARCHAR, profile_conditional BOOLEAN,
    attributes_json VARCHAR, source_oid VARCHAR, matches_snapshot BOOLEAN)""")
if live_macro_rows:
    _coverage_db.executemany("INSERT INTO macro_live_evidence VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", [
        (m["file_path"], m["caller"], m["macro_path"], m["line"],
         m["start_byte"], m["end_byte"], m["caller_start_byte"], m["caller_end_byte"],
         m["context"], m["classification"], m["profile_conditional"],
         json.dumps(m["attributes"]), m["source_oid"], m["matches_snapshot"])
        for m in live_macro_rows
    ])
print("Rust files parsed:", len(live_file_rows) - len(live_macro_parse_errors))
print("parse failures:", live_macro_parse_errors)
print("source files matching snapshot:", sum(r[3] for r in live_file_rows), "/", len(live_file_rows))
print("direct macro invocations recorded:", len(live_macro_rows))
print("The collector records syntax and context; it does not expand or resolve Rust macros.")



parser_process_exit: 0
parser_runtime: {'tree_sitter': '0.25.2', 'tree_sitter_rust': '0.24.2'}


Rust files parsed: 60
parse failures: []
source files matching snapshot: 60 / 60
direct macro invocations recorded: 4158
The collector records syntax and context; it does not expand or resolve Rust macros.


In [21]:
SELECT 'current Rust edge query' AS strategy, sum(expected) AS expected_direct_sites,
       sum(baseline_tp) AS captured_sites, sum(expected - baseline_tp) AS missed_sites,
       round(100.0 * sum(baseline_tp) / sum(expected), 1) AS fixture_capture_recall_percent
FROM macro_fixture_metrics
UNION ALL
SELECT 'prototype AST evidence', sum(expected), sum(prototype_tp), sum(prototype_fn),
       round(100.0 * sum(prototype_tp) / sum(expected), 1)
FROM macro_fixture_metrics;

strategy,expected_direct_sites,captured_sites,missed_sites,fixture_capture_recall_percent
current Rust edge query,29,0,29,0.0
prototype AST evidence,29,29,0,100.0


In [10]:
SELECT context, classification, count(*) AS direct_invocations
FROM macro_live_evidence GROUP BY ALL ORDER BY context, direct_invocations DESC;

context,classification,direct_invocations
conditional,other_macro_name,13
production,other_macro_name,1551
production,unresolved_bare_sink_candidate,39
production,local_macro_definition,6
test,unresolved_bare_sink_candidate,1968
test,other_macro_name,581


In [22]:
CREATE OR REPLACE TEMP TABLE reachable_macro_evidence AS
SELECT e.*, n.stable_symbol_id, n.qualified_name, r.hops
FROM macro_live_evidence e
JOIN current_nodes n ON n.file_path = e.file_path
  AND n.byte_range_start = e.caller_start_byte AND n.byte_range_end = e.caller_end_byte
JOIN current_reachable r ON r.node = n.node_id
WHERE e.matches_snapshot;
SELECT hops, qualified_name, file_path, line_number, macro_path, context, classification, profile_conditional
FROM reachable_macro_evidence
WHERE context <> 'test'
  AND classification IN ('explicit_standard_path_candidate', 'unresolved_bare_sink_candidate')
ORDER BY hops, file_path, line_number;

hops,qualified_name,file_path,line_number,macro_path,context,classification,profile_conditional
3.0,impl SharedParseCache::evict_before,crates/spur-graph/src/git_walk.rs,2094,debug_assert,production,unresolved_bare_sink_candidate,True
3.0,impl SharedParseCache::evict_before,crates/spur-graph/src/git_walk.rs,2095,debug_assert,production,unresolved_bare_sink_candidate,True
4.0,impl OccupancyCounter::decrement,crates/spur-graph/src/git_walk.rs,244,debug_assert,production,unresolved_bare_sink_candidate,True
5.0,parse_raw_diff,crates/spur-graph/src/git_walk.rs,3412,unreachable,production,unresolved_bare_sink_candidate,False
5.0,import_licensed_call_target,crates/spur-graph/src/store/build.rs,1922,debug_assert_eq,production,unresolved_bare_sink_candidate,True
5.0,import_licensed_call_target,crates/spur-graph/src/store/build.rs,1926,debug_assert,production,unresolved_bare_sink_candidate,True


In [25]:
# Validate the imported evidence independently of the native parser's point API.
_source_cache = {}
_coordinate_errors = []
for _row in live_macro_rows:
    _path = _row["file_path"]
    if _path not in _source_cache:
        _source_cache[_path] = (_repo_root / _path).read_bytes()
    _bytes = _source_cache[_path]
    _start, _end = _row["start_byte"], _row["end_byte"]
    _valid = 0 <= _start < _end <= len(_bytes)
    _valid = _valid and _row["line"] == _bytes[:_start].count(bytes([10])) + 1
    if _row["caller_start_byte"] is not None:
        _valid = _valid and _row["caller_start_byte"] <= _start < _end <= _row["caller_end_byte"]
    if not _valid:
        _coordinate_errors.append((_path, _start, _row["line"]))
print("validated_source_spans:", len(live_macro_rows))
print("coordinate_errors:", _coordinate_errors)
print("known_parse_raw_diff_site:", [
    (r["file_path"], r["line"]) for r in live_macro_rows
    if r["caller"] == "parse_raw_diff" and r["macro_path"] == "unreachable"
])
assert not _coordinate_errors


validated_source_spans: 4158
coordinate_errors: []
known_parse_raw_diff_site: [('crates/spur-graph/src/git_walk.rs', 3412)]


In [24]:
SELECT context, profile_conditional, count(*) AS named_sink_candidates
FROM macro_live_evidence
WHERE classification IN ('explicit_standard_path_candidate', 'unresolved_bare_sink_candidate')
GROUP BY ALL ORDER BY context, profile_conditional;

context,profile_conditional,named_sink_candidates
production,False,28
production,True,11
test,False,1967
test,True,1


In [23]:
WITH labels AS (
  SELECT source_stable_id, target_label FROM current_edges WHERE relation = 'calls'
  UNION ALL
  SELECT source_stable_id, target_label FROM current_unresolved WHERE relation = 'calls'
)
SELECT m.hops, m.qualified_name, m.file_path, m.line_number, m.macro_path, m.profile_conditional,
       count(l.target_label) AS matching_labels_in_caller
FROM reachable_macro_evidence m
LEFT JOIN labels l ON l.source_stable_id = m.stable_symbol_id
  AND l.target_label = regexp_extract(m.macro_path, '([^:]+)$', 1)
WHERE m.context <> 'test'
  AND m.classification IN ('explicit_standard_path_candidate', 'unresolved_bare_sink_candidate')
GROUP BY ALL ORDER BY m.hops, m.file_path, m.line_number;

hops,qualified_name,file_path,line_number,macro_path,profile_conditional,matching_labels_in_caller
3.0,impl SharedParseCache::evict_before,crates/spur-graph/src/git_walk.rs,2094,debug_assert,True,0
3.0,impl SharedParseCache::evict_before,crates/spur-graph/src/git_walk.rs,2095,debug_assert,True,0
4.0,impl OccupancyCounter::decrement,crates/spur-graph/src/git_walk.rs,244,debug_assert,True,0
5.0,parse_raw_diff,crates/spur-graph/src/git_walk.rs,3412,unreachable,False,0
5.0,import_licensed_call_target,crates/spur-graph/src/store/build.rs,1922,debug_assert_eq,True,0
5.0,import_licensed_call_target,crates/spur-graph/src/store/build.rs,1926,debug_assert,True,0


In [16]:
_coverage_db.execute("""CREATE OR REPLACE TEMP TABLE macro_fixture_evidence(
    case_name VARCHAR, caller VARCHAR, macro_path VARCHAR, context VARCHAR,
    classification VARCHAR, profile_conditional BOOLEAN)""")
_coverage_db.executemany("INSERT INTO macro_fixture_evidence VALUES (?, ?, ?, ?, ?, ?)", [
    (r["case"], r["caller"], r["macro_path"], r["context"], r["classification"], r["profile_conditional"])
    for r in coverage_rows
])
print("fixture evidence rows:", len(coverage_rows))


fixture evidence rows: 29


In [17]:
SELECT case_name, macro_path, context, classification, profile_conditional
FROM macro_fixture_evidence
WHERE case_name IN ('shadowed','custom','alias','nested_tokens','stringify','async_test','cfg_any','cfg_attr')
ORDER BY case_name;

case_name,macro_path,context,classification,profile_conditional
alias,explode,production,other_macro_name,False
async_test,panic,test,unresolved_bare_sink_candidate,False
cfg_any,panic,conditional,unresolved_bare_sink_candidate,False
cfg_attr,panic,conditional,unresolved_bare_sink_candidate,False
custom,app::panic,production,other_qualified_macro,False
nested_tokens,wrapper,production,other_macro_name,False
shadowed,panic,production,local_macro_definition,False
stringify,stringify,production,other_macro_name,False


### Design that closes the gap effectively

#### 1. Capture a site, not just a name

Add a dedicated Rust query capture for a `macro_invocation` and collect it in `extract/languages.rs`. Keep the existing call-expression and macro-body token captures. A macro invocation and a call-shaped token inside its arguments have different provenance and should not share `CallOrigin::MacroBody`.

Persist invocation evidence through `GraphFacts`, the artifact schema, and Parquet storage. A sidecar such as `macro_invocations.parquet` can be joined to paths by `caller_stable_symbol_id`, while avoiding an ordinary runtime `calls` edge to an arbitrarily name-matched macro/function.

| Field group | Minimum evidence | Purpose |
|---|---|---|
| Snapshot identity | graph content hash, source OID, extractor/schema version | Reject stale evidence |
| Site identity | enclosing symbol/kind, nullable caller stable ID, snapshot-scoped site ID | Keep repeated invocations and non-function sites distinct |
| Syntax | raw and normalized macro path, invocation origin | Preserve qualified names and raw identifiers |
| Location | start/end bytes plus line span | Precise diagnostics and independent validation |
| Context | raw attributes/`cfg`, test-context evidence, debug-assertion condition, execution-phase uncertainty | Select the intended build without hiding unknowns |
| Resolution | explicit-path candidate, local-definition evidence, alias/re-export resolution, unresolved | Keep uncertainty visible |
| Triage | optional sink-family policy match | Separate source capture from risk classification |

The two `debug_assert!` sites in `SharedParseCache::evict_before` illustrate why per-site rows matter: a single label at function scope cannot establish coverage of both invocations. The 58 invocations outside functions must retain their enclosing scope and execution-phase uncertainty; do not attach them to a fabricated caller. Include the sidecar in canonical hashing, manifest completeness, incremental replacement/deletion, and schema compatibility checks so it cannot silently lag the rest of the graph.

#### 2. Resolve uncertainty before increasing alert severity

- Bare `panic!` can be shadowed. The fixture collector records a visible local definition as evidence; it does not claim full Rust name resolution.
- `use std::panic as explode;` is captured syntactically, but a name-list classifier misses its standard-macro identity. Resolve aliases and re-exports in the **macro namespace**, retaining unknown results.
- A path resembling `std::panic` is a candidate until resolution establishes its meaning.
- `wrapper!(panic!(...))` and `stringify!(panic!(...))` contain similar tokens with different expansion behavior. Keep nested token evidence separate; do not synthesize runtime calls from arbitrary token text.
- Preserve `cfg` and `cfg_attr` rather than evaluating them from syntax alone. Use observed target/features/`debug_assertions` settings to mark sites active, inactive, or unknown.
- Treat source `.unwrap`/`.expect` labels similarly: unresolved names do not establish standard-library receiver types.

#### 3. Prioritize by observed benefit

| Order | Addition | Evidence for value | Remaining boundary |
|---|---|---|---|
| First | Direct invocation evidence, exact spans, context, and fixture regression coverage | Recovers the 29 fixture sites and six scoped real candidates | No macro expansion or panic feasibility |
| Next | Macro alias/re-export resolution and build-condition evaluation | Addresses the alias fixture and large test/debug candidate population | Requires reliable Rust namespace/build facts |
| Then | Targeted invariant checks with source-bound solver receipts | Existing guard/clamp examples eliminate modeled false positives | Hand-authored models cover only their encoded facts |
| Later | Optional CGA/MIR overlay keyed by compiler, target, profile, features, and source revision | Needed for dependency internals and compiler-inserted checks | Toolchain coupling and conservative path approximations |

#### 4. Measure success without conflating evidence and execution

Use separate acceptance measures:

1. **Capture correctness:** preserve all annotated direct sites, reject the negative controls, and verify exact caller/span/context. Extend fixtures for aliases, shadowing, qualified paths, nested tokens, test attributes, parse errors, and repeated invocations.
2. **Resolution coverage:** count resolved, local, and unresolved identities separately. An unresolved candidate is neither a proven panic nor a safe site.
3. **Alert usefulness:** manually adjudicate a held-out set under an explicit build configuration. Report actionable findings, invariant-guarded candidates, and unknowns separately.
4. **Graph integration:** verify round trips, incremental additions/deletions/moves, canonical hashes, source-OID checks, and coexistence with ordinary/HOF/macro-body call captures.
5. **Cost:** measure full and incremental extraction time, memory, artifact size, and query latency before setting a regression budget. This notebook does not supply performance measurements.
6. **Compiler comparison:** compare source candidates with same-build MIR evidence and targeted runtime tests. Neither a source-graph miss nor a bounded solver result establishes program-wide panic freedom.

This is an analysis and notebook prototype. The production extractor and its schema have not been changed in this turn.


### Implementation clarification: preserve scope locators and coverage domain

M2 review confirmed that physical Rust files and embedded Rust notebook cells need different source contracts. Phase one covers **physical Rust source files**; existing notebook definitions/calls remain available, while cell macro evidence is deferred. The artifact and query metadata must expose `source_scope = physical_rust_files`, so an unsupported-source empty result cannot masquerade as complete coverage.

M2 also creates exact `MacroScopeEvidence` records while validating sites. Constants, type contexts and other non-indexed scopes may have evidence-only IDs (`macro-scope-v1:…`), which must retain their source/range locators through persistence. M3 therefore adds `GraphFacts.macro_scopes` and extends `MacroInvocationEvidenceSet` with `scopes` plus typed `source_scope`. Enclosing and raw-attribute owner references must resolve to these persisted scope records; function callers still reference actual graph symbols. Preserve the existing validation entry point and check persisted scopes against supplied authoritative locations. Missing or conflicting scope data is invalid for a complete new artifact.

M4 writes a core `macro_scopes.parquet` table beside `macro_invocations.parquet`, with matching metadata, counts, completeness and incremental replacement/deletion. Both tables are required even when their complete captured sets are empty. Legacy missing evidence remains unavailable. M5 returns the relevant scope locators with filtered results and handles overlay scope replacement/removal; M6 verifies the complete chain and documents the two-table SQL join. The implementation paths required for this transport are explicitly added to the submitted plan; macro expansion, namespace resolution and actual build-condition evaluation remain outside this phase.

This clarifies the approved exact-scope/provenance requirement; it does not claim a new parser or panic-freedom result. Mutation `c1b2c0da-8dc3-4081-9dd7-b0c051e87721` records the M3–M6 contract before M3 dispatch.


In [27]:
WITH bindings AS (
  SELECT e.file_path, e.start_byte, e.end_byte, e.caller_start_byte, e.matches_snapshot,
         count(n.stable_symbol_id) AS exact_owner_matches
  FROM macro_live_evidence e
  LEFT JOIN current_nodes n ON n.file_path = e.file_path
    AND n.byte_range_start = e.caller_start_byte AND n.byte_range_end = e.caller_end_byte
    AND n.symbol_kind IN ('function', 'method')
  GROUP BY ALL
)
SELECT count(*) AS invocation_sites,
       count(*) FILTER (WHERE NOT matches_snapshot) AS source_mismatches,
       count(*) FILTER (WHERE caller_start_byte IS NULL) AS outside_functions,
       count(*) FILTER (WHERE caller_start_byte IS NOT NULL AND exact_owner_matches = 1) AS exact_function_bindings,
       count(*) FILTER (WHERE caller_start_byte IS NOT NULL AND exact_owner_matches = 0) AS unmapped_function_sites,
       count(*) FILTER (WHERE exact_owner_matches > 1) AS ambiguous_owner_sites,
       (SELECT count(*) FROM macro_live_evidence
        WHERE caller_start_byte IS NULL
          AND classification IN ('explicit_standard_path_candidate', 'unresolved_bare_sink_candidate')) AS named_sinks_outside_functions
FROM bindings;

invocation_sites,source_mismatches,outside_functions,exact_function_bindings,unmapped_function_sites,ambiguous_owner_sites,named_sinks_outside_functions
4158,0,58,4100,0,0,0


### Approved implementation and verification gates

The user approved the first phase on 2026-10-31: direct Rust macro invocation evidence, with **SOLVE PRE → TDD RED/GREEN → SOLVE POST for each task**. SPUR worker: `codex`, model `gpt-6.1-sol`, effort `xhigh`.

The implementation plan is `docs/superpowers/plans/2026-10-31-macro-invocation-evidence.md` (commit `350d976f3`). Execution plan `6888d1b0-fee6-47eb-8cc7-c8f18d8ca570`, epic `bd-156`, has sequential review gates:

| Task | Execution issue | Deliverable |
|---|---|---|
| M1 | bd-3dj | Versioned evidence, identity and provenance contract |
| M2 | bd-1rr | Direct AST capture and conservative context |
| M3 | bd-1k5 | Full/incremental artifacts and canonical hashing |
| M4 | bd-iea | Parquet table, metadata and completeness checks |
| M5 | bd-3u8 | File/caller queries and overlay/backend parity |
| M6 | bd-2u9 | End-to-end regressions and reproducible query documentation |

**Current verification status:** M1 and M2 are approved. M2’s physical-source owner guard fixes the notebook-cell provenance regression while preserving notebook definitions/calls. M3 is running with the durable scope/source-domain contract; M4–M6 remain pending. Full artifact persistence/query coverage is not yet complete.

The research counts above remain prototype measurements. This cell will be updated only from observed implementation test and solver evidence.

**Local build update:** The user then requested local builds. `SPUR_REMOTE=0 scripts/spur-cargo test -p spur-graph --test rust_macro_token_tree_query` completed successfully in the default sandbox: **4 passed, 0 failed**, wrapper exit 0; build profile 43.18 seconds, test execution 0.08 seconds. Plan mutation `ed862b95-9d08-4220-b1ed-b9c199cd6a28` updated all six task bodies to local verification while preserving their dependencies, Codex model/effort and solver/TDD gates. This is existing-regression baseline evidence, not an M1 feature result.


**M1 attempt 2 review:** Typed evidence and contract tests were produced, but GREEN stopped at a missing `GraphFacts` initializer field in `memory_eval/memory_graph.rs`. Brain accepted that same-crate mechanical scope addition, preserving prior work for attempt 3. Independent review found the fixture line values one too high: actual invocation lines **3, 4, 6**, function lines **2–5**, file lines **1–6**. Earlier PRE receipts (`sol_464b1e08c2c44f3f`, `sol_c6fbad0d1cc0486c`; rejecting controls `sol_b3b6d03c0d374e62`, `sol_03b385b54c424b49`) establish only the supplied finite constraints, not that those source annotations were correct. The retry must refresh source-bound PRE, demonstrate an assertion RED for file-scope evidence falsely marked `FunctionBodySyntax`, then obtain local GREEN and POST. M1 remains unapproved; M2–M6 remain pending.


**M1 approval:** Attempt 3 supplied a compiled assertion RED (10 passed, 1 expected failure), then GREEN (**11 contract tests passed**). Existing local regressions also passed: **4 macro-query + 11 memory-graph + 1 store test**. Fresh source-bound catalog PRE/POST receipts were reviewed: identity/ranges `sol_d8d5392abbe54013` → `sol_130b6917aed3460d`; context `sol_91d5fb4c75354c98` → `sol_77ba56afb840475c`; invalid file/function phase `sol_a1c47a86e6e14129` → `sol_6d06935d4da04ba1` (UNSAT in both). POST uses the actual GREEN fixture rows and additionally checks digest uniqueness. These are finite-fixture checks, not parser or panic-freedom proofs. Beads completion audit 96 and worker evidence 90–94 were independently verified; no active signals remain. Authored test commit `735d1dcf67435b9dc0837ecaab7fb13a0ffe0f9e`, fix `2035c09f83b239b66058ea92a9f530fb8da94a08`; normalized worker tree `72ca468c1` is identical. M1 approval unlocks M2 under the same local build, Codex model/effort and PRE/TDD/POST gates.


**M2 attempt 1 review:** Local RED demonstrated six missing-collector cases (including 0 vs 42 sites) and a missing direct query capture. The required GREEN suite passed **16/16** (6 collector, 5 query, 5 HOF). Fresh catalog checks cover the 42-site finite contract and expected rejecting controls; selected PRE/POST receipts and actual logs were independently inspected. The brain also ran the extractor on the real **60-file spur-graph corpus and recovered 4,158 sites** with valid physical source OIDs/byte ranges and caller references. This matches the prototype’s source-site count, but artifact persistence is still pending.

A corrected local notebook fixture exposed the boundary bug: one Rust cell emits a row for `review.ipynb#cell:rust-cell`, whose OID and span refer to virtual cell bytes, not an addressable physical graph file. The source-addressability regression fails. Mutation `80f4bad2-ab90-458a-ad5c-153916ef59d9` requests a physical-source-owner guard plus RED/GREEN and refreshed PRE/POST, preserving ordinary notebook definitions/calls. The combined debug corpus-and-verification driver took 110.865812333 seconds, including extra file reads and hashing; it does not measure isolated extraction speed or production latency. Scope locator transport into durable artifacts is explicitly flagged for M3 preparation.


**M2 approval:** The new notebook assertion failed before the guard, then passed after gating by actual `NodeKind::File`. Local GREEN is **17/17 capture/query/HOF**, **11/11 M1 contracts**, and **18/18 notebook tests** (the latter uses `TMPDIR=/private/tmp` for the existing canonical-path harness issue). The physical 42-site observation remains byte-identical. Fresh source-bound PRE/POST positive receipts `sol_0c9c76d19900490a` → `sol_47e6bbbd095f4c2e` pass; Cell-owner and nonzero-cell controls `sol_f581cc4bc7ba42cd`/`sol_9f5572432b474c4d` and `sol_2dc04d7c94e74453`/`sol_ef95396171b54a4d` reject. Actual logs, source bindings, scoped diff, committed tree and Beads completion132/worker audits125–130 were independently checked. Author test `6a1eb31fdcecfdc4746088ec1a7da8d6c75178bc`, fix `59225ba135c9674e848c46ff20a80f839b2c8fe2`; normalized tree `a6c6c6eef` is identical. Approval-time overlap warnings were inspected: M1 is an ancestor, its only shared changed file retains the intended module addition with the M1 API intact; the warning label was cleared. M3 now retains exact scope locator records and explicit physical-Rust source coverage as specified in the design clarification.
